Copyright 2026 Google DeepMind Technologies Limited

All software is licensed under the Apache License, Version 2.0 (Apache 2.0);
you may not use this file except in compliance with the Apache 2.0 license.
You may obtain a copy of the Apache 2.0 license at:
https://www.apache.org/licenses/LICENSE-2.0

All other materials are licensed under the Creative Commons Attribution 4.0
International License (CC-BY). You may obtain a copy of the CC-BY license at:
https://creativecommons.org/licenses/by/4.0/legalcode

Unless required by applicable law or agreed to in writing, all software and
materials distributed here under the Apache 2.0 or CC-BY licenses are
distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND,
either express or implied. See the licenses for the specific language governing
permissions and limitations under those licenses.

This is not an official Google product.

In [ ]:
#@title Imports (run this first)

import itertools
import fractions
import numpy as np

# The kissing problem

The kissing problem asks how many spheres can be arranged tangent to a given sphere, if they all have the same size and their interiors do not overlap. The maximum such number in $d$ dimensions is called the $d$-dimensional kissing number.

In dimension 11, AlphaEvolve improved the SOTA lower bound from 592 ([Ganzhinov (2025)](https://doi.org/10.1016/j.laa.2025.05.002)) to 594.

In dimension 16, AlphaEvolve discovered a novel kissing configuration of 4320 spheres that didn't exist in [Cohn's database](https://dspace.mit.edu/handle/1721.1/153312).

These configurations and the verification code appear below.

# Kissing number in dimension 11 is at least 594

For $d=11$, the best known lower bound was 592 ([Ganzhinov (2025)](https://doi.org/10.1016/j.laa.2025.05.002)) and AlphaEvolve improved this to 594.

This section contains the description of a sphere packing in dimension 11, discovered by AlphaEvolve, which shows that the kissing number in dimension 11 is at least 594.

The description contains a set of 594 points in 11 dimensions, all with integer
coordinates, such that any two of them satisfy $2 \langle x,y \rangle \leq \|x\| \|y\|$. This implies the angle between any pair of points is at least 60 degrees, hence a valid kissing configuration can be obtained by normalizing each
point.

Since all points have integer coordinates, the inequalities $2 \langle x,y \rangle \leq \|x\| \|y\|$ can be verified exactly.

In [ ]:
#@title Data

sphere_centers = np.array([[10519816772536543232, 42804977090115584000, 50416112064832430080, 14973261118195425280, -74689919198934384640, -34277624995045175296, 19876361167799787520, -47323492356155686912, 45633842141050183680, 74234934210238021632, 9368067945243893760],
 [71704104522096852992, -13072106356574392320, 37608544100392058880, 29402301267578204160, -6618164164596896768, 11319320869395841024, -4542741755992301568, 83937675207888732160, 78148015056509812736, -26905797711912628224, -10848218850978318336],
 [88711464331963777024, -4655684889973334016, -35135055444404178944, -70462868868369489920, 23445554555200950272, 29753236027998261248, 67513539304611840000, -20042984505543766016, -24707501986433921024, -19590061802119098368, -17268604208148287488],
 [-12103627765004855296, -10453521717615579136, 67145443164953894912, -31560722395101696000, -10520776754279460864, 6122890131483601920, 44756682421852528640, -37699749070972469248, -111349874659754721280, -6803900615474478080, -109301500825756448],
 [14425087184875921408, -21550381719035576320, 25119750571358982144, -79945484134245597184, -57725729577478414336, 1893333457934633728, 16735034709820618752, -70389169742381039616, 74140680224558366720, 3880907636070020096, 918699815383488256],
 [-54367717321058549760, 88978584581755633664, 20511525513134542848, -56617694827074797568, 42584657686061211648, 19132038330121482240, -42190536662811680768, 34455909233924075520, 42673626886300459008, 9263940296258879488, 15005005440738705408],
 [10190714470868963328, 32714888299700617216, -19495183638863708160, 83529087287211114496, -30339571163829805056, 1223158648477468672, -48324366659533791232, 60619976861407879168, 15483277693852651520, -39110961707060133888, 67419399136306774016],
 [-49962576798164746240, -22677489867748614144, 19215853707275755520, -31820917842670383104, -18411103488936972288, -99123046381182173184, 65415465258591543296, -43618942550757761024, -32075444118867185664, -3257176854768828416, 372012875922802688],
 [27214812391947440128, -63738651655689863168, -12003241605504356352, -6112406721609119744, -85778700430082064384, 89983864515143696384, 5807742873228286976, -9912238723498389504, -33246368931618869248, 10042133381195454464, 375852664779230528],
 [-75111843225744703488, 27920954852175474688, -14464529403030050816, 47405739564465717248, -6223170974073914368, -63711164814436433920, 33814408844549529600, 47934373541064785920, -22823257010277285888, 12994058022920278016, -68348157400790343680],
 [106758529022233296896, -24907481001765670912, -46071774544454131712, 5573134403324518400, 10559654895891304448, 1050673741022778752, -45367194918332080128, -54186807965695287296, -37717861471616344064, -28188264736697487360, -16805326994305456128],
 [-1054762446890440320, 84283027997113286656, 48231774056082374656, 46213405536478371840, -62763983792095371264, 10115686512781142016, -70882261642465509376, -30031944131610595328, -3432902440418654208, -3739611597149693440, 14755348110605944832],
 [-22440032707251363840, -13204313376119830528, -91406567924966326272, -9199422906734849024, -1037346011755212032, -96038629682646106112, -12296482208538738688, -8854311060988686336, -55187114096930758656, -12666978475255820288, 453650180090014144],
 [14057215763964989440, 25858866195923648512, -1782313313669196288, 118585033481650077696, -1886467618600074752, 65976517683085942784, -23608846699103776768, -25549103455404322816, -35740497465956581376, 1089003172801601152, -1017515528544505984],
 [-24787795907414626304, -2208651365155614208, -95180090449010098176, 42692986703244836864, 78013356435001802752, -37504359247503900672, 24792220005741932544, 31137622348069707776, 22644543782563979264, -23921938303071498240, -10719927527433674752],
 [-37407696077451198464, 8360380365798836224, 64617494197121474560, -77329563389159096320, -22182445057507422208, 32265932031581298688, 11528522566037460992, 32841439828572413952, 64803257489340530688, -52952913960162148352, -20136194966175354880],
 [32718779110980255744, 36485513097239261184, -35163435312447389696, -16091760403398500352, -48135214761145106432, -28828537831217549312, -69718368084451573760, -19602205235108401152, -38540745478201819136, -57751439042938421248, 68318005013564432384],
 [-9385321150103646208, 7994009026361363456, -47837004062052409344, -73119083003163361280, 52636407654787186688, 86126169617993531392, -11149915002535530496, -34388838951638581248, -49929902768823599104, 1702185945714546176, 435674543462709376],
 [13516543011145922560, -72258174539955372032, 22554287107003006976, -21832872089556770816, -35659859709219631104, -49903287067285340160, 7967747605930125312, 11427829504306952192, 78488622999401398272, -72199749191319011328, -7710312718251177984],
 [-9180089655630985216, 25245239082902798336, -14971337576581795840, 29651024415583444992, 100920697804977569792, 12214337598447163392, -32536305613487824896, -38720180412467716096, 26011766027986767872, -79420944568866390016, -8764239391400372224],
 [43979331258682064896, -49956033960885215232, 25935522046975197184, -43783421303038713856, 85748442645258076160, -21613191877842239488, -51902066216387256320, -58662873390568136704, 18563645872570920960, -14402417400295049216, -15859509600595533824],
 [15383359859108763648, 12300648818101442560, -48415326463604310016, 43677467042123857920, -68116083367926546432, -79704150541179994112, 64616977331223764992, -11699752420675852288, -14816539715537825792, 39170481627891154944, 8927817434082267136],
 [3988627132888394240, -46119603801199796224, -9162628995186410496, -42565523237320572928, -10964137268566620160, -94637128166487162880, -89290312912613179392, -8630900708280621056, 22788020437857058816, -9693049038213337088, 1570702816842403072],
 [-61039589124445052928, -75832380300501712896, -20669214110524121088, -7423548975994947584, 85898267685228281856, 43567916249735536640, -6569882682448139264, -47366366682031742976, 11462055502703929344, -6422213817474441216, -9128490216023657472],
 [-41946691644292423680, 12076590277323274240, 85503563516646391808, 57323680679793614848, 40351454479963619328, -4978843542326627328, 60976267349533728768, 35625682528496418816, -51029014294668550144, 5528063759372362752, -997889575302298752],
 [18861541563068039168, 21741976531968667648, -51841986847515869184, 19720866810584104960, -22412072658417680384, -54380687258292355072, 46337829975924891648, 103332318859430002688, 39770126170068385792, -2504996084984649216, -913950209398908800],
 [9645306040096796672, 39397201296529317888, -18000886090326267904, 79740666974397595648, 79337197163791089664, 1113465559763763712, 48872416282611671040, -22082879520949661696, -4862720869679984640, -9517152385847568384, 64531191453330784256],
 [-21053450026082725888, -22925722591675293696, 43551214824397193216, -2492132655364401664, 37892468818734612480, -70752272803574964224, 86023615770098057216, 51776286652608806912, 42320587335858774016, -21866314796277850112, -10704133116198694912],
 [1773693742269357056, -43486807914645561344, 27411821135305183232, 31185311617756557312, 128216517861626839040, -23353022722422788096, 9839819860338939904, 30532492873695019008, -7831213784828383232, -6034146122524386304, -6946819748646626304],
 [10644071394676207616, 75961108744866922496, 59932663128408842240, 6372661848828860416, 93670103631433875456, 15262357319443978240, -26465908192232628224, 15831339872072982528, -46028048010630283264, -13592724161038911488, -838700775632881920],
 [-88890205676868599808, 4824017744594316288, 35141867051016921088, 70355959803887886336, -23559591244258525184, -29776778144403611648, -67656714650908073984, 20055579158707445760, 24771399307183542272, 19739999222422536192, 15612873815090415616],
 [-15418648442397044736, 980347919893848576, 43477015697891229696, 8938908501726090240, -6853203940257743872, -9207408251590211584, 32955287571219668992, 2935543457167084032, 134461550806576939008, 16214197798600044544, 26097857533889024],
 [-37957436216274870272, -34043926453496643584, 76340972649511927808, 14758451947909861376, 29387563559972229120, -99615991240382578688, -28313177131707457536, 26994399438090903552, -28240823548915482624, -4167504460230139904, 582383230407749376],
 [37957980485923274752, 34043495479379243008, -76341011890933170176, -14758283799707260928, -29387130462212374528, 99615962938454818816, 28313511930007605248, -26994517181572911104, 28240891078607028224, 4166999344107379200, -576490753065896832],
 [-86778174992572465152, -28734161858605285376, -10554747127245942784, -33994025311706988544, 17078188253070321664, 11891627123559962624, 12119861123020027904, -31657130132185202688, 23942106886016339968, 99417147647126880256, 10762776892436830208],
 [32195727948340932608, -8329425641224560640, -64298147342123319296, -50553208973657300992, -8586919005253198848, -1117941942806691584, 64199056121427271680, 10045676930344640512, 95169635103547850752, -22802304713379192832, -10374228361405806592],
 [-3237920328741338624, -64891009096240390144, -27673731263312384000, -43680607548528680960, 13930464456602068992, -12518105268778418176, -14394667769407918080, -8434369884097043456, 84059947780291837952, 49684937797073158144, -68695994757766873088],
 [-6076290556693559296, -62611723476346847232, 68857438323997515776, 59292495995423612928, -5613419735249616896, 69377736617505505280, 43272764708001890304, -40746625824687767552, -30198207141903974400, -12805221904499288064, -10510444422081476608],
 [-13510133548565088256, -43951490257315774464, 26059466659793887232, 56098890828844441600, 51248577813242109952, -57746273676191645696, -49932017510353174528, -14714921888610201600, 81336628532920287232, -21435089230367576064, -9578026005279528960],
 [-41432830674584387584, 76981690788604657664, -53200457459373629440, -20306875053138890752, 54468412989159350272, -12393636935865686016, -33442333502031142912, 19083836413286486016, -40566341886068056064, 181160908459383296, -68146165076184449024],
 [-2321497510472356864, 16216131947917393920, 12858644715737655296, 1510560940018095104, -88931640335587786752, 32216936124912021504, -58190984251896406016, 55997655011167625216, -75074102357579169792, 1318780628117623552, 15347214420441677824],
 [-8025380618153481216, -79333641727341363200, -50645464446362279936, -32912621377242206208, 6766714185965217792, 70373139382182060032, -78411894721929396224, 20393511096837214208, -67836723914749856, 14385669215026214912, 1481400532819170816],
 [1370669334565022720, 66828453906316500992, 27632279240003952640, 43057293684728381440, -15802192220880936960, 12765475742341705728, 13910742241808971776, 9064785596913930240, -83970071506876563456, -49718468747448311808, 66943500958234656768],
 [-93337339258589249536, 624721200533385472, -21081628886104846336, 84972342070614818816, 17191927082121156608, 33806356665257394176, 30035632445308416000, 7127994302923504640, -19891858616049274880, 50165716952079253504, 9870333812333572096],
 [-43930171622488317952, -76152852476587851776, -33210409421854879744, 53851839063326130176, -13532411344383858688, 37225705683830874112, -36606962548556390400, -48774374805152620544, -67883541789101973504, 12216339627942107136, 836561123005360640],
 [9152223827593072640, 9453195027307001856, -18756931150041939968, 74593847175004012544, 38937814696026464256, 71708048801739997184, 28392906426284683264, -26975018205184692224, 85003789519685615616, -12182555199262453760, -11082219116378368000],
 [32449438432970362880, 21461153521568022528, 53606462614898040832, -71013194587460272128, 55819209396373479424, -55527219967997968384, -58435390924900827136, 40137773648343228416, -439208267952282176, 31902403538468003840, 10259110186245609472],
 [-30132526901211365376, 5242650866058221568, -14504711523480823808, -13900066068190330880, 2664219810759826432, 62717182043603443712, 37740005827378503680, 77619904403403063296, -15654870507530715136, 93373219068916793344, 9256609761191757824],
 [-37859164353114382336, -12223941672392941568, -47929439742122360832, -260229727415753280, -7890436688088037376, -105245984301686145024, 20658586206213328896, -5919030943731297280, 79274397508763811840, 3547195850207821312, 480084388403426112],
 [-25819278037883281408, -41735379446001721344, -6925197562954140672, -6111198241800090624, 67898450117140037632, 25019967472025378816, 13146715766544674816, 50329624248905031680, 88837671007234621440, -58147342168736292864, -8302930653285856256],
 [-48314390092692856832, 80083622565443207168, -47543098739933192192, -27171390273078173696, -18070124249368868864, -9016301353271031808, -61358291890380259328, -70350532045177110528, 25897644932720779264, 1576573246736655872, 15386353793248581632],
 [-2586657811127726592, 112267453764505485312, 19207072670044651520, -16431916451111047168, 6906851895822127104, -21194800399375478784, -18427166869470685184, -16887655598697265152, 7567127914331643904, -83021045156829364224, -21378659204783509504],
 [-66114247472889856000, 21111093376725999616, -11137403127111278592, 54147472996712366080, -2816234699416935424, -77609989108119142400, 29332347851089764352, 28698975513686753280, -16621243244211738624, -36381197180217741312, 67750957717518524416],
 [58570682888447459328, -49712695069042688000, 58207408503211327488, -61895766026494156800, 41846472691352141824, 12821799919966466048, 49119554901785608192, 9186085639403866112, 64621321033302581248, -4893095499876845568, -6671438566557398016],
 [-28828259746715856896, 73326759176006819840, 20951406644736020480, 30689773547763294208, 28871269394199785472, 40641514429645668352, 25055341051727876096, -8423626180090975232, 55975576135165485056, 88299771843545858048, 8674119826238131200],
 [-9345669657738512384, -53317121746773835776, 29777275387309793280, 97571321389736296448, 17103249758993815552, -27467007624057294848, -18676452425129041920, 11071095327281168384, -9721716562497523712, 43947295002043899904, -69978227935599222784],
 [1940868633809186048, -25990842014941917184, 13688863648452425728, 42865731320210759680, -42242042432602832896, -44354354895157813248, 68662528093256843264, -81987622981940887552, 56686412918715645952, -23696998960825794560, -10526895595006406656],
 [71144982850934259712, 12414925768912623616, 21207452461715542016, -59963951830098739200, -72246111506491146240, 52758651870672764928, 42414379522093121536, 38862865400887017472, 34636432671432437760, -2173325119853165568, -466127154683114880],
 [26109647053936123904, 40831541359818178560, 23786501371574575104, 56542320453143740416, 8381887391380620288, 31839058423290601472, 51608190245888417792, -68943592580208025600, -7186288956527502336, -83770278252937232384, -9828692218613196800],
 [42577839263555911680, -4401843985353282560, -36332032448357695488, 14255780234198851584, 5701079377017503744, 48472004889814482944, 39918541150338686976, -78237848686986133504, -62019462903295942656, -13837777335403309056, 65482766320333496320],
 [30804642414659223552, 1501887013023946240, -62031315857639456768, 24561005806346768384, 53810587884293472256, -63460465726531133440, -40868433138366742528, -43751612477661995008, 33618358345582034944, -18783513170883977216, 66223151432415354880],
 [-6759158317276132352, -11287162328059774976, -15303873740737091584, 11838888678662537216, 32932727651989786624, 48258528492955475968, -91094058129767120896, -65632595081431900160, 71579920051979968512, 9310819552586985472, 1012911534687232768],
 [39341561234570969088, -55183125314837110784, 15700763404934393856, 7011534938026786816, 43642422356920614912, -35187898096690348032, -34896879973606109184, 50251191303649918976, -80994552714402594816, 55143385680820879360, 20068032156829540352],
 [35268337309426692096, 2909518546148685312, 57047129192027430912, -34607089386608914432, 4332436802764619776, -86148652621691142144, 362282567676580864, -33723662750583435264, 45187311801600286720, 24285082656441339904, -69744458996336410624],
 [48705275036972441600, -789758168231108352, -70199112918145310720, -69163719034994950144, -73283695507025068032, -43280270989428441088, 30117847616886185984, 30007161204095598592, -20550195988603318272, -14839182797701494784, -14468639086200010],
 [41236582140495519744, -29153731131466252288, 74158854974407016448, 32128624453166514176, -17924200764271085568, -65558608838932299776, 49714339698493480960, -12126458231179724800, -45119738845110239232, 54895698722296111104, 20218510295588786176],
 [-36027931526339268608, -65943460830441168896, -66535918881208393728, 31205429329703960576, 29983210655258148864, -25797967377583210496, 51315017353590775808, -45370765960397922304, -7405896340250240000, 4264590073628065280, 66343144367870263296],
 [-6188068446624012288, 15648841529726289920, 642258192085574016, -5315359991400269824, -10379344053905997824, -5062584444738588672, -9671773926146740224, 2254867289285433344, -5787507458643432448, 56581092022778699776, -134161559712533692416],
 [87008934845191438336, -69062105120107175936, 53944721681955274752, 31280910216729636864, 2649711265337053184, -57560797776012705792, -40240078646165561344, 7625895549716219904, -434958377606028416, -22547073508543885312, -6570365265224020992],
 [50994775389419446272, 29597714830683795456, 8955619039004945408, -2984981612406638592, 6840701891675415552, 25618657957964374016, 79319844985754419200, -54805421566591811584, 62652347659168972800, -7735366003037561856, 65030231286116663296],
 [-28481469350882197504, -4308235913160603648, -23337001550315778048, -38639783105373233152, 59611967451467677696, -67869228870479233024, 6874224947366469632, 95939166967027548160, -38400117863653318656, -4969117856009360384, 95958375101378000],
 [55000347708180176896, -29080557107982868480, -80237661016509022208, -16872611694753372160, 81820434748629598208, 45883714965739298816, -12087381272831655936, -11172383639797622784, 44941792071321321472, -6977831887621443584, -6731015245204945920],
 [34331239593968816128, 108495737740999491584, 17562508926843244544, -20164056662942584832, 17393031029035143168, -36023546619055206400, -18555113671185473536, -24487441328597680128, -10212537926785435648, 64083478775947608064, -36850411307179683840],
 [6041703215849489408, -8895869756124616704, -68069670308434108416, 29440303215390502912, -60649562852646756352, -28169376518451380224, -19170673175394824192, -104793330393747046400, -16787278042069768192, -7696953065728889856, 354251233769329920],
 [30173873750931705856, 11013580204495763456, -5642384598965059584, 109420006638799290368, -13493127982397927424, -11692462525604986880, 74650133496068407296, 37738514845595557888, -19023960855428173824, -30329346435827650560, -12014119270137421824],
 [47936907308190998528, 39426082353437835264, 41334845621183938560, -84169971114351771648, -37671022735382396928, -42124249964167356416, 60099184900401938432, -27065612238621323264, -25097861308789186560, 29606623751454408704, 9385219137299310592],
 [-22483527192770199552, -47737970677553356800, 80896946616833196032, -75282111519726616576, -13034126889063708672, -36316780457054437376, -24379301514504663040, -43329954273981595648, -13476585497283784704, -50699573108560805888, -8320997560586944512],
 [40028031547809882112, -3842038191129566720, 35040530976372506624, -1244517443222224128, 96697279784824274944, 73061355950060175360, 37386402402559016960, -50054852671650463744, -4195288259448104448, -4722504218315334656, -17326851410696105984],
 [-94448155678200823808, 14441291231543083008, 33732008285196255232, 15943936166048788480, 64470734593577943040, -4168681381318079488, -27899947046905671680, 53856374275261423616, -7712427159077254144, -54102821753281691648, -20436304042791890944],
 [24514659597958193152, -101420658935251648512, 5827168699882352640, -7245733893811016704, -39796978138379599872, 9570710297514682368, -56880280167323295744, -70214585566386470912, 18973403119359197184, -22971868262442754048, -8714775616382827520],
 [62003133797873270784, 37699183251025977344, 6214583142763800576, -30241443479032496128, 28719419232455651328, 65012412932814807040, 9893050742714693632, 280785762964046944, 60552003325449633792, -81591235280038920192, -9286741396630853632],
 [-91514803722900226048, 24057644346049191936, 2607337220572451328, -14560408273953415168, -3865581537427572736, 8080750980414656512, 12305857897616316416, 51297668741607604224, -96769551493756436480, 12106915842244849664, 15214746566017667072],
 [48408605819409956864, 6298565294753444864, 41694551500812075008, 63855900119749935104, 63369227091921977344, 50537713605420785664, 9492977368387835904, 42905352412330672128, -16806910137505421312, 66416138116123828224, 19614124542458867712],
 [19935803799508557824, 72573059117818961920, -49315196867368280064, -38335565254708322304, -44481550199969144832, -2561848118191950336, 55151550006541746176, 21564876884940414976, 35359185188561723392, 73454511670043222016, 8627634413936423936],
 [76299363986906759168, 14369584772180013056, 7204787602733115392, 75609120585446408192, -31208290432001916928, -41317518647964852224, 14479600937788755968, -47476422183540482048, -2689562570672225280, -24167865335969275904, 65155673461365112832],
 [64521070471263993856, 21050804253052706816, -23516916407030190080, -81583986609734778880, 42307318782048575488, -23664018118270341120, 4348695386160786944, -37220998223689646080, 54626282432067428352, 53180511470520221696, 19914014941189509120],
 [13492773174380767232, 43935143448791842816, -26026441934830555136, -56001164772944699392, -51272651339438120960, 57727944630726950912, 49908829270188048384, 14724100587908505600, -81399428109380403200, 21558963367315886080, 9615978759871623168],
 [-32582852054689521664, 9621020007005052928, -9385541171794839552, 64124957968082321408, -2413930614864626176, 4035800482688166400, -36743727405239586816, -74031443693449150464, -11848693546130991104, 24443380797183090688, 93984175702023061504],
 [-89190433819822080000, 7846419042918196224, -10264021835568607232, -16055366867619389440, 85097212536628363264, -24096337304008495104, 70484223699736231936, -4681866574310588416, -21683684017358413824, 10776938687715274752, -86478262437722000],
 [-34219059617352921088, -107637028240605773824, -17642508667357837312, 20645123635343380480, -17481836884076795904, 35334633356058570752, 18808265509294821376, 24674188102299119616, 11063142197535928320, -63345340938106568704, 40400193963724414976],
 [111744360516093329408, 25249231772282290176, 72251497452983459840, -8187059857171382272, -2424006913384399872, -4523780084075412992, -5140334400987712512, 3090641634171466240, 25307153871234887680, 47701078757221105664, 19704780803159887872],
 [31877107523157200896, 61193652079754190848, 7703815240446374912, 8165697060219928576, 3426647805310925312, -3084151055141858816, -38635080024068145152, 92170563172706320384, 39240556710609436672, 72558434738377187328, 8794731190118948864],
 [34959973397853892608, -33942807347187576832, -23122590916738633728, -2201530888214250496, 19232460130091945984, 13719678090165592064, -81820475482759708672, 19017163800605368320, 44062336971554209792, -87580086394694877184, 40473889537629315072],
 [52590371760195952640, 63884941196282642432, -25570874409667006464, -50951132596142612480, 53318681184493846528, 20239989610309468160, -87442544656763387904, -19794254957738323968, 5001294555712753664, -19119740629063200768, 153367175419722528],
 [-2006636919355050752, 25974959383339405312, -13748446732142714880, -42857881277399982080, 42256416633168551936, 44328830845187825664, -68630437748567916544, 81949293825013121024, -56730946448765157376, 23843979413881618432, 10493770871125422080],
 [-39326988906511171584, 33363107940171362304, 108563669459453329408, 7987381913736351744, -6327465108856424448, 40752266458052050944, -14090989922380341248, -17365509759076575232, -26559801357774860288, -62379945905084162048, -21700511447419785216],
 [43937282363114602496, -77705959358827823104, 53177278254109630464, 20681434047180038144, -53137049292985999360, 11986200350310922240, 33980900371755122688, -18755626497080864768, 41092672078029324288, 1957900961349054720, 66239676541950844928],
 [64832867522116255744, -12828815586247645184, 39414277179004461056, 62911929646568800256, 50922386150101622784, -3864955125904747520, 15643521669078335488, -18574793289386094592, -88743652464101965824, -22655016715304538112, -17673157180196165632],
 [29817672680276668416, -15331069649599946752, -6819662445289872384, -51060560664276213760, 3668949412853390848, -3807139783003064832, 40931221290990321664, 82106943506016436224, 2140395809844993280, -89370910275122593792, 39542838568225251328],
 [-111635850357988245504, -25261651230732140544, -72255815114797662208, 8178233167429686272, 2460671894762690048, 4467161861097878528, 5143446890622844928, -3124114186117317632, -25333681993645211648, -47330640278670401536, -21103566761701249024],
 [17872948260784091136, 38354339808904069120, -9388954480462772224, 42924505312303865856, 16272525813811558400, -39925133937671151616, 30697299917179551744, 6565321943861094400, 74360022101282914304, -90367316728010948608, 38948739240576434176],
 [14046131809028876288, -86585439374470135808, -3314155684283380736, -10593583032045146112, -11440627099367108608, -78070949022531633152, 33458143478238826496, 45161479201208164352, -10220359615843278848, -7098650247018485760, 66488476736598294528],
 [-632636491196587264, 66831689078769410048, -26208677832684113920, -103136437721459163136, -17634246916700641280, 11596037087995371520, -3276457473054987776, -14676103522226812928, 27223427839044349952, 2040502634389699072, -67604292809977094144],
 [43930190531897573376, 76152780835244228608, 33210406032476631040, -53851811859956465664, 13532456718124906496, -37225679820530868224, 36607010342245761024, 48774369834091839488, 67883572295764418560, -12216639968972986368, -835728893808882432],
 [-8084167440047634432, 3157083760230721024, 62648336637495279616, -62654679338910531584, -5431043533627730944, 21239964126918098944, 11858915761166749696, 9094985089681782784, 57627023021714702336, 30428137095738597376, 94348276927108628480],
 [56915775191938236416, -11755359550263416832, -5272398021751777280, -36356487372296556544, -61192649336530051072, -56696238452664868864, -14518968158679326720, 34530335350777344000, 82288068354933506048, 42595927472264577024, 10148776773071073280],
 [183297384512060896, 5665566233955068928, 36083901448758001664, -56821493049418760192, -25311251586365108224, 93089034087155646464, 53908128316972089344, -55595608386312790016, -9835561875233378304, -41468936503289028608, -9778167588157937664],
 [-47243736774544908288, -4230272132745120768, -95767075132854616064, -73379656802551046144, 44744543235240951808, -19119565562437885952, 9508118926725621760, -40308239864725921792, 29344699217785114624, 5250874431863119872, 908507904910004864],
 [1266280276795964416, 68046575779082838016, 35340047357425692672, 44752642297619587072, 26166434702318731264, -22114743441121222656, -12681745850030057472, -86026619301110415360, 71647065780359872512, -5073468969443212288, -470522155738065024],
 [1046378116015772672, 50699769535006203904, 2581413745910695936, -58121847266851864576, -22094220762445086720, 51884176337679908864, 8860706370426311680, -81709393401806635008, -4344603184847077376, 75972456285049667584, 9795304722468096000],
 [-12003419887127187456, 11365076482287521792, -57124933638101401600, -46579139468838027264, -47799712093562560512, 492606430390662272, 93728451992529911808, -70613813594515316736, -3834062207360558592, 906859902612048640, -198003024310284608],
 [-39547944299821547520, -24530758251386126336, 85224981885498195968, 39406414879345672192, -27542449595495178240, 31000505020946436096, 2463767535723138560, 8730330524992118784, -5375496548395996160, 94156905152779943936, 9989244179240601600],
 [-18688371802639548416, -21495709859973898240, 47383556823489593344, 34639847421922754560, -47715266272682090496, 63623609191948591104, -77996963205491769344, -10067693451499667456, -12207480335248443392, 18558693869780316160, -67220439720226267136],
 [-57481652212797063168, -19762578019711315968, -45543639311360524288, -37755672493918208000, -36032200768332369920, 50120772307634012160, -31137530238347395072, -69133403030192439296, 7113248224585717760, 22805227731021987840, -66735272080663216128],
 [20996408980150480896, 22923522646217113600, -43523062870465642496, 2514587912707109888, -37868214419797123072, 70813416361499672576, -85973337731954638848, -51781570267172790272, -42318005741637672960, 22039336062434287616, 10645228231913474048],
 [8882886958639413248, 11087984786288220160, -60230229594692845568, -16124745609296216064, 61306657835729526784, 38119888159669731328, 78298573989984927744, -31034336991056982016, 193886519069617920, 70572514055527964672, 20064599569208020992],
 [-35904890483813208064, 3180978646805792256, 17434723662821414912, 86764181783723425792, -20298447625606995968, -33146859702800560128, 41805692499271524352, -69168537046245695488, -67815463547967258624, -2168066305006700800, -646167918822072448],
 [-40815725054921523200, 2930087551405207040, 36959654985944711168, -13527124786718253056, -4571663622437665280, -48474024204168765440, -39061677824293019648, 77660071653305024512, 61920484067207643136, 16120854046361720832, -67327577111291412480],
 [39707838591328337920, -100732946863218933760, 7204155375650157568, -32288335812355600384, 37409090684564611072, 43650062046663172096, 2822316124494043648, -25035572524570529792, -70177938593675558912, -7623303981009354752, -6101212836914493440],
 [-38223236849631633408, 37437440676179615744, 62686364360892006400, -61094479760117391360, 20687255205015801856, -42706182431962963968, 26749398418670137344, 24464946449661771776, -33042043221003923456, -44120598443109744640, 68184526512814481408],
 [75874198912076513280, -27409642711851372544, -67161351916376195072, 17670542289475061760, 29465190892913364992, -37254897100040404992, 34276902554609668096, 53553909351385653248, -58836002014727438336, -25046364936234950656, -7677632142710247424],
 [-23192122638756474880, -10916092686527750144, -37264500820485062656, -57597121317911838720, -31549073430571982848, -65678373875489087488, -5642153778285931520, -41444677099926650880, 7493498310947880960, 17336145183010301952, 94867180891097120768],
 [50272276634396311552, -14447297110258243584, -34189045211607728128, -24379701047749574656, -71425832151635214336, 3865671180492957184, 19437721229593739264, -48103672949691678720, 15064444555930040320, -29793734808994222080, -91410061053238542336],
 [-98558476700113392, -5676652658624025600, -36134148571713359872, 56806745100935913472, 25369608547137429504, -93112916932615880704, -53926945601238048768, 55482140185345449984, 9863260077051799552, 41463485519721840640, 9830269174470326272],
 [9354065557177976832, -43683323307588239360, -51957746796577554432, -99825904610440003584, -26622214086025891840, 44544162843344830464, 28911546954786430976, 1906509244103653632, 12430340897566781440, 57604596090451304448, 10596301485887248384],
 [36408846059918856192, 79265979285387427840, -38390794710646480896, -6711046672199854080, -33203585794876702720, 12048286535119972352, 85270079304155627520, 22865356536848056320, 1355974661584312064, -58967167367800668160, -11404600241698383872],
 [-20862643561555689472, -61376303761470095360, 26078612213993893888, -6467557933930553344, -60274499185368539136, 1350287946621036032, 33290272635304161280, -90080224427710496768, -25928668610833235968, 56683092078561501184, 10538083522359846912],
 [70054844196437688320, 18753862436408356864, 25218443381177741312, -50581696754194219008, 52722342046696660992, 5348591389294467072, -16008906772135219200, -48251354835667935232, -75070933271052042240, 43286339998524096512, 20545598683336241152],
 [40607592533715410944, 12865704396509478912, 51044642632512487424, 51757935457654300672, 69808322653862346752, -57301352779410259968, -47580272028558450688, -35775847371109031936, -9347481559508107264, 50075214890568630272, 19590459316901117952],
 [36913743101647462400, -16642046736302583808, -78849605846137569280, 43469083757278208000, -7209617217530789888, 47807611666097045504, 19612827396505686016, 54741019333867577344, 32534154861097582592, -71886876064290914304, -9310480984331374592],
 [-20487589643156918272, 38602252562863161344, -103532825768743272448, 17100149292235657216, -41921787719551508480, -7381861019994005504, 19965605664951160832, -6431219482873071616, 26665772580879454208, -36116721080115023872, 67744667782106882048],
 [-27615421131234242560, -102019841573218975744, -12931803226798499840, 5421751193622330368, -17312712687450140672, 36883956319541510144, 7214002711944684544, 17704573360594253824, -12185930506916409344, -3828335890256556032, -90675145094789300224],
 [-4007021725542121472, 64319127909836259328, 25286498318817386496, 94949527396575608832, -16889765090526531584, -36258473017516433408, 3079365546360735232, 22942778449486655488, -28503156188832305152, 70368867711194013696, 8381123640815415296],
 [89323227422647877632, 12082209085212035072, -19156646362991308800, -17402435435984963584, -3478603259522063360, -100584899349989490688, -17466386547637839872, -5771341493726624768, -29905880993919205376, 35289243682080206848, 19394142238155165696],
 [50736801642075963392, -38674888212956766208, 41236627004353331200, -55671583323437088768, 52841397159439917056, -69834317886323482624, 39244473849342222336, 6968466743731869696, -52981754052343865344, -23720413915128901632, -16758851503172671488],
 [9465177914077396992, -26147707695274582016, 31841709635826839552, 20786777003879182336, -24647140688655269888, 44651701474813902848, 97279438967879499776, 20111875911044182016, 55639248539688886272, -62500932426673815552, -9363192937362841600],
 [-85159642464955056128, -58168127312006201344, 9972532389736304640, -25209773753940258816, -7457784469302639616, 5845503483258548224, -71846616450881691648, -2900428389204382720, 52641240107175952384, -44159835754703233024, -20641297291063062528],
 [-55014919666603581440, -74920257224125399040, -8878785260057930752, 52488684821739773952, -43484423195267112960, -73899734587227045888, -19858210104425385984, -27910220220454060032, -3279794820457558016, -41137163680283262976, -8721941799954900992],
 [62822976871545724928, -28277776338020233216, -25882834840461000704, -24095173768735092736, 63323856330189643776, -75961524768424394752, -5543459370761016320, 44696117962281836544, 58368089769501499392, -16918553478957051904, -16721411527296884736],
 [52747279841650868224, -64418854054416015360, -13510163213622429696, 59285596632869355520, -11042526103201978368, 42239457617587986432, -50899344153541910528, 49516308662841720832, -56733242003999383552, -29598907372644044800, -10258042163460544512],
 [-24531054887843024896, 101427572336187277312, -5810518155237200896, 7260358519472980992, 39789076538539671552, -9652253829475846144, 56846788648316936192, 70153729686574645248, -19064155720039329792, 23064954641089261568, 8797586129067327488],
 [29843512727151808512, -22530666956369289216, -18357310652899665920, -88884299136581943296, -50872065074027487232, 11100305343570061312, -16220459552344569856, -73325267810191278080, -60320815682083700736, -12333425779744493568, 894205221769662976],
 [860976964896961536, -84137752233401712640, -48174165410556731392, -46310947914816585728, 62677369643093721088, -10037002309539926016, 70707768944227532800, 30082999599553732608, 3418625300694251520, 3920113293622754816, -16489115168711557120],
 [18729443579827363840, -37823543180168044544, 102987169692352495616, -17630118609867352064, 40536895575225892864, 7426575459139156992, -21011393211534589952, 7401888176189227008, -26711515871194796032, 36408290433893425152, -69611432914021228544],
 [9102471308516063232, -45594940405584887808, -10457862621746040832, -9010670597231647744, -48911056781642670080, 78513332326328401920, -20708169156951773184, 6025062715801836544, 97510568920443977728, -28479190226341724160, -9198860093571387392],
 [-19740602133172953088, -50626905349380825088, -74902719789483835392, 23372172924484968448, 7202796985016811520, -3016582998756300800, -6029923902869391360, -54528535091262619648, 72120240521411239936, -65840852550639427584, -7846816532852506624],
 [-10478114344213035008, -42837711892780261376, -50414722041496100864, -14955236782105911296, 74725077786714767360, 34285194310013583360, -19844665308299890688, 47320688153263366144, -45623677916666421248, -74338781125766742016, -8231846893588096000],
 [4273742468685862912, -1011805770495717376, 654983084624054912, -19674858261008916480, 61008874836164141056, 21358669288901046272, 48053784554846093312, 39332488866439299072, -84957581143692197888, 33732687949084893184, -70576784595257458688],
 [-66321921581985931264, 66103592189334790144, 5389449459026653184, -17709232878340696064, -57296128948210122752, 2988211823839608320, -39541130076579848192, 17493881154936702976, -11790442772916463616, 81371150038083518464, -27304486049302433792],
 [-13211280089093521408, 59680085789704339456, 115392597059295756288, 20572934488995393536, 13040855547735525376, -29854777107366522880, -17903421416107732992, 24422792886351953920, 39661403727649087488, 25584442450916802560, 9856787022754910208],
 [-26221238798196277248, -27026414851834499072, -57783441188235722752, -30140591394863316992, 27082524563411230720, -3070372829036227072, -85574918869542223872, -63990588374680256512, -33152410656347922432, -52753460512086605824, -8363450213111546880],
 [-61716938836695293952, -38604835077889409024, 10655217096635283456, 80673029917706321920, 47562861074022039552, -8141888252955745280, 54857547349040447488, -18889969557980999680, 21090897220926988288, -60327023628451446784, -8843809717628414976],
 [-58500715186303590400, 49662885550824546304, -58260113826292195328, 61974788114982420480, -41795233755980406784, -12857570600024260608, -49120878145732698112, -9241457547353755648, -64684861429601853440, 4923965080096551936, 5951449849519761408],
 [56720205800267415552, 33965141450325057536, -3912018602825043968, 19981492956423016448, -14519917100833189888, 50865271691435343872, 25679464421090738176, 109252055299540631552, -39504162757764538368, -6053703063728078848, -1385070357801290496],
 [58977041498483482624, 3868603276457814528, -46240069492883709952, -56840841565456449536, -30735034032880132096, 65954084673510940672, -40212173022631690240, 34633455581473452032, -36368449743566049280, -58213260277005074432, -9657233323888508928],
 [77048188344428527616, -29117981330235187200, 14149363852636960768, -47309019907465568256, 7630318327913664512, 63957448719304548352, -32678872905967620096, -47766165334864609280, 22782812976612204544, -10792548756535230464, 66495229088522387456],
 [26287188091359617024, 27009372546533228544, 57787443053885472768, 30170215715428495360, -27007301075817132032, 2985301012866177536, 85577214000714153984, 64014150944595705856, 33117334140039221248, 52734641239705321472, 8404814737302891520],
 [46305654742132383744, -54980576182273687552, 55857643650113175552, 19357102387279556608, -58934872243500851200, 9571771144773158912, 18276212860630243328, 5121525973573809152, 48269519140492771328, 44436387272263573504, -69903457365407334400],
 [23802184831637917696, -13634042660079220736, 49711967693543587840, -118324651015742177280, 9777911798045245440, 39269048126410080256, 2950316458269817344, 31468353970610167808, -43533942920682078208, -4637847322303214592, 544433878667146112],
 [85316194140771844096, 58170933025055047680, -9993364080017907712, 25176365936908140544, 7498942521402988544, -5927350236542452736, 71848187181827399680, 2866305921859560960, -52671216540673605632, 44760566741640257536, 18496853954604060672],
 [-35546904181155266560, -88807227699338772480, 59978400753162313728, -55534689321007742976, -10608352764505950208, 67287399965035937792, -701262817929518976, -14369356207800629248, 23043617850234028032, 23794742984825229312, 1398389218683971328],
 [-17973409421472663552, 38480705594074365952, 26948596413312688128, -23651331911573606400, -15130327483313747968, -102249449011518046208, 26721818444230508544, 48522753274835345408, 7173799126158382080, 69248463216468762624, 9457702303303675904],
 [-44089794041239797760, -60203514185303646208, 97357617488859119616, 39018836614991257600, -2442266470368096256, 8890265760998498304, 1093490599839166464, 47716882895082569728, 44708944705205706752, -27563752152906842112, -9326029973050210304],
 [-24329053152407302144, -1671311957483424000, -1796878529328905216, 24054573065128108032, -73415470863264759808, -39702976846187716608, -55674696579017523200, -38305321283084607488, 66252668472697405440, 12730532316496144384, -67207993357108420608],
 [87830852845890502656, 79480761308954705920, 13070893022169468928, -24150955758665809920, -39225860027710939136, 39850954675963166720, -3222074283209008128, -50100273173758230528, -28177125152597561344, -23461786982459129856, -955403606931342464],
 [13237742992533000192, -59673731926863732736, -115411385848409767936, -20663137558259974144, -13029683446323644416, 29807208236669587456, 17848025704630528000, -24377622295021953024, -39728101720654331904, -25483546305634926592, -9814790960424689664],
 [9385341642893694976, -7994059075786298368, 47837001591273594880, 73119108587212201984, -52636371027920420864, -86126149612005818368, 11149957258084663296, 34388836397427081216, 49929924571086577664, -1702413252581874432, -434403827602869760],
 [22471873815420391424, -7251300877533774848, -72661328158134353920, -8197963409331548160, -74804891636611809280, 43159925116389163008, -24686947884814888960, 76335903240543338496, 26037002030658781184, 34513352646920937472, 10457836204137396224],
 [-49118213985200373760, 14652227534716225536, 34243985283111804928, 24875681131491983360, 69835796998922919936, -2297159090333738752, -19003688206318653440, 48000882581318246400, -14695862766183053312, 28362773907412254720, 93762353979272331264],
 [51792584105615695872, 23473590787470790656, -22101174752270163968, -4949565671719523328, -24082525663137062912, -56896365787769102336, 49161807135888080896, -53375662897591844864, -77238329364649721856, -53499927410391973888, -10982677980432920576],
 [28598175604931485696, 24214619997542576128, -6080686518776481792, 5190944567540643840, 22015064479033167872, -56798016050960654336, 46162739748083245056, -106386110057874259968, -13171661571043174400, 50867387486101733376, 19730254319227097088],
 [67810756877742538752, -50894478491490770944, 39041554220541599744, 45661147983468732416, -15962239716850718720, 32700016778337927168, -41753604639982723072, -45682563562304184320, -42579940219446902784, 59949368305908154368, 20469954202596229120],
 [30158249882130759680, -5272571520827816960, 14500929587650772992, 13924993897745037312, -2632514984791422976, -62720037754006298624, -37704033366308306944, -77620738999489921024, 15662019122929219584, -93472849411823190016, -8172519592537522176],
 [98048169150471520256, -12673540215173402624, 12704792142408034304, 2664612512603993600, -29222340111061213184, -56356831610754842624, 78707270758357041152, 14352662856471842816, 25192816366993580032, -21250332190540611584, -17842977262527797248],
 [37859129407885377536, 12224031650819164160, 47929444250147069952, 260196286426015264, 7890374823066549248, 105245957400170545152, -20658646308595068928, 5919039995095303168, -79274432608916013056, -3546856257214178304, -481045644747895680],
 [-30765197686451818496, -21888210705698971648, 35172641336162283520, 39821253336630468608, -28337905983268184064, -26571252196878168064, -2982464844126400512, 118055154026545496064, -26774080994409758720, 35439181341757140992, 10092748665278044160],
 [71728266152682938368, -53144918963329449984, -5140668618345991168, 12998106955936548864, 52399336156966100992, -11387432870839150592, 31918676089404162048, -18292755100531015680, 3992649959602878464, -1192416030008207616, -96784367066866892800],
 [-27147981919182704640, 25229676686774906880, 8511236605101151232, -62705372434360459264, -43183983788825296896, 109125075463838859264, -36372083028808830976, 24558718522004963328, 9439488906710149120, 19295846514221916160, 15475329417390030848],
 [-77178608928627605504, 21154358288515338240, 60634067784393883648, 7718776097730669568, -14228143584516251648, -64501454478744117248, 6559061591791806464, -23288340100581830656, 52708972868055310336, -58766741689052200960, -21399666557220081664],
 [14548634539947669504, -46177711486579081216, 11334606205682987008, -57191726262557335552, 89407387529942417408, 41696086934912106496, -13102875370390882304, 18800587918691270656, 3510406007577324032, 73321168202811817984, 19721868837907398656],
 [-57987042760858714112, -102011808719863463936, -31428668475152711680, -64733088817098219520, -11646631502386921472, -28750868952013856768, -12997841632247873536, -23225180429789364224, -32142927145723715584, 11127208489666983936, 1854542628159500544],
 [-11802953937445728256, -31721550360707067904, 19627862916526743552, -83730750811241332736, 28303061322481414144, -936974176453825664, 47679223883573436416, -59682755025094737920, -15748239354517942272, 39349608569627369472, -69301758954680287232],
 [-24612832494378356736, 46556999262406246400, 2629376498991759360, -76477547312325574656, -48193130299141652480, -2769864093369035776, 7549464669508354048, 103057983276076023808, -6381030447640422400, 7923182609899001856, 6108212049705976832],
 [6758749435207358464, 11287547183945742336, 15303904905341630464, -11839030505600086016, -32933082753362100224, -48258530478916861952, 91093777480751644672, 65632686888852881408, -71579998067319029760, -9310180796603197440, -1017222467311233792],
 [-18862024098575908864, -21741524468362121216, 51842023880320180224, -19721030322730573824, 22411655829854437376, 54380688260596908032, -46338155986333310976, -103332209334864297984, -39770215945753796608, 2505720127512141824, 909229310636007936],
 [-23802170482656534528, 13633987093356302336, -49711970426355212288, 118324671685786828800, -9777876807230316544, -39269028207829549056, -2950279977334756864, -31468357822327635968, 43533966560334430208, 4637615047841449984, -543839076449376448],
 [24080747100066570240, 3609442556472416768, 82481929972629209088, -15557314235823718400, 58061665675582373888, -34674811269771964416, -18541586350226468864, 27237816632547688448, 32376663844945379328, -85778581544408367104, -8629031105775509504],
 [-48664690212172414976, -30233759627973550080, -8425262422830273536, 3093096198022139904, -5489105887190077440, -26071861498206879744, -78391209254443433984, 54746639833430032384, -63234031657551822848, 9592315486658654208, -66875878419321896960],
 [83589353016083251200, 55375675145872449536, 4507305587887109632, 40902642323796811776, 4525275772085656576, 3544193030112545280, -69840633875764191232, 26077298946670276608, -61726446464473112576, 23485022524433870848, 8865925390230085632],
 [-43798976526723776512, 30236692663804129280, 5593207476496556032, -319538890327633088, -6108539322152611840, 6272519242249479168, 89463338936004706304, -19961738305233092608, -31675515037262868480, 27218737561402449920, 93225859033728090112],
 [-7922423836511059968, -40410548197759107072, 18680111068705431552, -79762966278209404928, -77631312653475708928, -2102108030292668672, -48212233274486366208, 20985798354892353536, 4559857369688524800, 11446811266451812352, -66509641549427343360],
 [-10743313463350294528, -97781174786657255424, 64338718239233761280, 8646225644904009728, -56391050460902268928, -9631201540273623040, -22505365996195737600, 17082592535463434240, -61488395960580726784, 5875883314629250048, 949078215032052736],
 [14932327819475134464, 31446320072905330688, -14641482675844698112, 47418039265851842560, 6327689679719931904, -44779294794551033856, 29400750783648153600, 13280115190261813248, 76816592307918127104, -24807126954134847488, -92202004605815586816],
 [-36869854953117810688, 16610925753121970176, 78844655229913874432, -43429374892797870080, 7268281556737384448, -47802233191909580800, -19562309101669580800, -54754701821873643520, -32524793980696285184, 71769400375289077760, 10621694063033102336],
 [47664833078211674112, 37522253110647103488, 82827656416967393280, 6693379729387275264, -59955690681975873536, -8202441756609765376, 10575859440888852480, -22940181657870327808, 59995371850555662336, -56454860536787697664, -10311747538141968384],
 [-56435892799742656512, -43028789868800688128, 13493118869908594688, -47399032516015357952, -26660732110962212864, 63214864935352623104, -34866888318718816256, -1625708366217590784, -70102308728892612608, -55520163341520224256, -19767987964672184320],
 [18016238752325853184, -38513724569397420032, -26954536326818238464, 23670120388907008000, 15162289772142651392, 102251065905598791680, -26692762815396843520, -48530616781920296960, -7167393740308152320, -69356959871991930880, -8288470214754000896],
 [-34319658287033896960, 17828515379494447104, 12581804764322668544, -35403414576694894592, -78369779168791248896, -39773229598626480128, 63961378357004148736, 14276519835097389056, 45004113682755526656, 14869734811975526400, -67864458453587959808],
 [49766384465085849600, -30633883003172147200, 49985197502264688640, -30358713414367010816, -3111432841494209024, 61360593281590591488, 71145563027430948864, -11502637408304517120, -29627642733318057984, 68555801260939116544, 19906579172467265536],
 [72759666260404781056, 33536182550207471616, 22788379129995792384, 69950540403246727168, -33084849473297432576, -30305349801664790528, -13510831807816349696, -49123329593485729792, -5367305728969889792, 17684334974218690560, -70688254543913459712],
 [-48792705965302571008, 48380452964742062080, -49922040216690040832, 19967340160613834752, 24613118695345750016, -56977867862797950976, 39872049463936663552, -75733797423217803264, 4541939481928537600, -53501136327870693376, -10251735688913229824],
 [49796359577615286272, 50547246576635600896, 28125619822154072064, -12275539382076827648, -1794055822339671552, 24963453155341217792, 74906240096465043456, -40243294655575277568, 42579844236161884160, 32829905205469986816, -71090959699128991744],
 [30771754323092226048, 21944665509320101888, -35212362590207807488, -39789951494642016256, 28359176397190508544, 26474008780246953984, 2902937447238598656, -118088174789990252544, 26790270871720255488, -35321784978362130432, -10136288771435292672],
 [-29543144086246309888, 79968338370359279616, 24854827126041153536, 7577495393624665088, -3166901871981162496, -57782538746092724224, -64017941385636937728, 65878499296670662656, -41850331039420194816, -8723418356974085120, 14848748588035606528],
 [-1006916892259819648, -50736387055807520768, -2584876280919564800, 58142342154658349056, 22130973577936625664, -51879227821089538048, -8825739225164131328, 81702725459751632896, 4354990175862359552, -76074705852061073408, -8652918124364317696],
 [9476578036666361856, 29734937947878850560, -99679334792770863104, -53398026612432076800, 30224427871046176768, 31745646227828858880, 35187443731127324672, 68943685562853154816, -10159603981250390016, -802445423999682560, -478376911899289792],
 [-52906639736092942336, -32828273368042430464, -38497941430311002112, -30022301843444285440, -38068140116281991168, 64043661251628998656, -2115522002494884096, -65470039297907564544, 23731205393892524032, -18262769222747332608, 69134355811915636736],
 [-65002479372702130176, 13028622442732556288, -39417759842814435328, -62975190847502671872, -51093404653836738560, 3874981628648842752, -15724784807792082944, 18627030129049649152, 88714402981927862272, 22859906814681747456, 15835481547533088768],
 [59652114024842100736, 18716393654900133888, 45164112333415350272, 37925283617854111744, 37818098058430431232, -51083303713077501952, 31009812767200260096, 68508331413641060352, -7643895796153099264, -20896162218018078720, 64818809347134996480],
 [5734610589651226624, -10736221709155960832, 61579082749824868352, -8769234461437319168, 15703783040910149632, -3655726596610002944, -18410193565229645824, 76337247064533106688, -89170936045345030144, -57307008460621365248, -9960103073405458432],
 [4068852761396836352, -33934507181013880832, 70665923404284796928, 31907714337158287360, -27919467214447792128, -43337294690277285888, 39402978553402548224, -2073595020381071104, -27393242764288335872, -89891513868553830400, 39405632683707277312],
 [3481127031315645440, 14083025175368935424, 56272841200316407808, -75030548398049181696, -10696982006074873856, 9922943357292029952, 11785537184750739456, 27450365081899139072, 44739905072826884096, 91563241235016925184, -35785582007469662208],
 [-58924300868906385408, -58886221265656274944, 54514694037276532736, -26779540226722496512, -57149501438326521856, -59173638785459625984, -8914046209041526784, -1607471734510391808, 48941241950665777152, 42062625686675283968, 11630850531402850304],
 [79206006803747536896, -34365437786047143936, 64527669659563589632, -17029788315343296512, -6799178819790426112, -2001019904826158336, 32351800304172089344, -88999896348444033024, -14606412764968427520, -18735607557942874112, -16954028503433431040],
 [12131356718396729344, 14991964992937213952, 25459443422191718400, -62055831088701005824, 10160317858519490560, -34153347995372679168, 75244400477449994240, -43370733286983434240, 28501815276967501824, -84878419457749843968, -8826670392375411712],
 [31968734594571943936, -18284233843272798208, -46671305603799867392, 17481518263967080448, -38623299939465248768, 23489995843245797376, 87492161213849419776, 37058657492451442688, -9976213795306356736, 41797428589853466624, -70849365460768260096],
 [7781001460538201088, 25008767259821535232, -5344492860190253056, -85152848852725792768, 108504397326637285376, 81885498626756192, 20267751760383741952, 15684594857140934656, 12424106654901090304, -33503083149340741632, -10281223628470179840],
 [-3988659385257726976, 46119608885052612608, 9162630528580980736, 42565518139087863808, 10964121493931356160, 94637137358740668416, 89290302543801794560, 8630906396800240640, -22788015409168027648, 9692994083850573824, -1571127335090308608],
 [17058713741956872192, -25551885937494745088, 14787317497673633792, 17783449284304420864, 35013403440640065536, -10012896316607172608, 128249216209227759616, -36727513664370978816, -32352602685216096256, -8296642281841382400, -7795104949300920320],
 [-52466089133509361664, 17696657135584524288, 28059141578266689536, 9648279053196277760, 69887210147999899648, -26628062194096173056, -25995850936166322176, 40214211082048405504, -19592449142493200384, 90895475011194847232, -36051494659174821888],
 [-30158716340096778240, -11003076857721044992, 5614732265026087936, -109455104595137363968, 13450444669167224832, 11718496473658640384, -74569481860832919552, -37730894513497841664, 18989173159036502016, 30461218310083436544, 12023739251939870720],
 [10186919964348534784, 31879422242664517632, -15340458259849011200, 72456016549065269248, -59170667859586269184, -4100013037627423232, -44448007147127709696, 69955641658213056512, 6393054965822182400, 6788223478004285440, -68545242558972461056],
 [25866109988001656832, 41707205298674950144, 6907034849025419264, 6120965297333016576, -67849236025151471616, -25036365514092933120, -13085620423470680064, -50335014448434241536, -88827662480299294720, 58039057017671606272, 9501588639186663424],
 [-36347097075651493888, -79331851706335625216, 38338641518661877760, 6673899431786274816, 33232463294307188736, -12068131490853132288, -85234619610624393216, -22877152658177261568, -1261512984777157120, 58979708732452511744, 11423072680240936960],
 [-71445574399350808576, 11404362443180972032, 3912172823753020416, 61323008886193569792, -15718141880183298048, 22294886751885025280, -45275654082304237568, -79959041881485017088, 6690168221009460224, -55858074349726490624, -20381054250869755904],
 [-39684043976326193152, 100751172290578006016, -7202311444192124928, 32300853598063001600, -37393858135462363136, -43685777829513650176, -2817935830391936000, 25021530539306418176, 70201011497744924672, 7658164497247334400, 5443888584910466048],
 [16971652273102217216, -80386255692217139200, 23312955266759536640, -65839017841456709632, -53142058717652803584, -3442847746929814528, 79074232243811155968, -3727208591807611904, -20048596116102778880, -23414545428947992576, -9852370587019016192],
 [14960153691169912832, -58274213278198505472, -50007105302431973376, 72271511932887310336, 60741121253147287552, 55928138697208799232, 47117806965616197632, 4544051590284345856, -32347641176835948544, 4993319419089385472, -17464809121066567680],
 [-28734204266510295040, 33749680206608830464, -108262508454648725504, 61284010226253352, -47550399899830132736, -17532468440157769728, 2387866755807973888, 10896943294125103104, 25760447955492458496, 11003329693022709760, -67934191517972807680],
 [13760468954690244608, 25855594687230279680, 6428154207372550144, -25187263265418534912, -20907020828298481664, -117355555852354404352, 6973994069006438400, 35100631547659476992, -2326042759116152320, -68202489905586888704, -9868067756835248128],
 [-47667313335854080000, -37607702297535897600, -82806942079717785600, -6637107479239195648, 59962426375662870528, 8125341395926196224, -10493316459544463360, 22871326711643009024, -60005230564269047808, 56472664982603718656, 10296018019617607680],
 [22440175644667564032, 13204162869749518336, 91406556400188882944, 9199479446803254272, 1037479107904326400, 96038638745837764608, 12296591806780987392, 8854278986218610688, 55187149704665956352, 12666669440931497984, -451861837620947072],
 [-49579340731211538432, 12019991681486522368, -82905198116182622208, -71890156167602692096, -44236571833300000768, 13094243087428282368, -48665068216935571456, 15674931564773447680, -45744819022004199424, 6628530349528827904, 15973794521792612352],
 [35240455576213340160, 15595477639788158976, 38642285171432448000, 26800304665686323200, -92544556812940132352, 19611496834392768512, 84220015079358611456, -30305734744670846976, -33178960114855247872, -4342124178330520064, -1108412653350260224],
 [4045055767808849408, -64353006912718553088, -25280621627969400832, -94930587166227791872, 16935049311445956608, 36257271216585764864, -3058899086058943488, -22956624477762068480, 28513644820512108544, -70466877912639045632, -7288030766586522624],
 [-78966374649923895296, 32420740742360645632, 21408828795125829632, 2756688104141483520, -29097599031818129408, 9236042089326878720, 90728955855867166720, -13418498160404254720, -32386968217875607552, -56398741297768243200, -21256374357703049216],
 [-71694713197623820288, 13035778976453754880, -37546647285301460992, -29402686586433744896, 6506955186652838912, -11297482822395375616, 4565747287515380736, -83970775064809177088, -78120399574003023872, 27075002961196380160, 10769217051779751936],
 [-73945946479030304768, -15242938437782366208, -6808549672393012224, -75324142038947463168, 32842687659316568064, 41297648831164006400, -14461099445129134080, 46816044374040584192, 2751976262291899904, 26232744092391575552, -66909678013488521216],
 [-22456024293882986496, -33125615582106173440, -61993274836261371904, 24241362145105039360, -82624502661021237248, 28583132908876349440, -65332203774461009920, 1565367230556250880, -3639372825371228672, -58244363613942792192, -20627171540657975296],
 [-49658416432435847168, 30626369195827507200, -49992913045575999488, 30344356408182894592, 3151282442764237824, -61411836484257505280, -71148211483450351616, 11467116470421473280, 29599065774501543936, -68185809122062712832, -21301433999556947968],
 [-40012623545318793216, -74272794132143898624, -7560476031698096128, 27390983281321660416, 33455111057312079872, -39827141679432171520, 39676849853672620032, 17353295323853010944, -92556646032239165440, -24560071177794572288, -10120373743462709248],
 [-2514863200072954880, -21305067846388768768, -23064469853925273600, 49324643462863331328, 46721995869072400384, -45780295978851074048, 75067302430034362368, 52966768259696033792, -24495782091642003456, 68246090307256270848, 19391229803644141568],
 [58871469809456619520, 17840566303433414656, -16779434596668553216, 18447917938006122496, 74272184804262264832, 18713135321572659200, 83742152534210772992, 53302356897742340096, 35609843184267722752, -7247786849486884864, -18194533268341342208],
 [15201874243393859584, 24537841551636545536, -50755826247180722176, -8576859867996559360, -18852052774160441344, -26647747206155309056, 7329331194314962944, 73324368174545190912, -54375831410804498432, 10914880797256445952, 93400226754735341568],
 [-25068765391183654912, -54411523352167751680, -85052199022884962304, 73571914085054005248, -35944707025237274624, -17154512156604059648, 9731034657227849728, 54558557514168991744, -28113624140905570304, 9710887766592751616, -76307873891286864],
 [18045309342433032192, -20260826353833721856, -10943886231788185600, 76019586295372267520, -12850060596247470080, -28660608700163092480, -112899312228338745344, -34180461980438470656, -12952068780943237120, -8603373516455680000, 551098877455524992],
 [29952664513288167424, -63119380801661493248, -8888394066627191808, 45400773599277965312, -23853327874182606848, -72546304969891545088, -21094483052206276608, -51602203823281479680, 36693860395749998592, 63498095029031739392, 20939427237557448704],
 [-37414448953855631360, 32127836457912246272, 62039980017079189504, 78554863423611666432, -30320164468882198528, 19808747954228023296, 34532765318767812608, -10482084558852308992, 39088096925587791872, -25440505481135120384, 67357430490905296896],
 [39592465524840267776, 24489365755488800768, -85227856452487512064, -39384370182561169408, 27583577165855191040, -30996215624313729024, -2425602945759422464, -8736502662440437760, 5387576539902754816, -94265559908602527744, -8775652953762297856],
 [-19949409981331304448, 12802892432231227392, -61687793833836806144, -19181251043112378368, 46887201742228480000, 51480522620717490176, 54714794493359415296, -19955388528432975872, 13537791325130741760, -14207022960053139456, -91745205380992614400],
 [-36300745143804559360, -5458167339960733696, 9770240521794574336, -8241489727523763200, -69407785902876712960, 70249141194016481280, 69462132367345229824, -13830984339651670016, 52568664421380087808, 51286913107739426816, 10231205355372476416],
 [22508378729857486848, 47726481384831647744, -80840011068367454208, 75359571857614061568, 13105362013636913152, 36259848534418575360, 24448519755950895104, 43274184023760388096, 13466121877635155968, 50707720529923358720, 8362611687088165888],
 [37427554812613230592, 33070035254180966400, 25063068892964519936, -45383443010765406208, 37848877866264223744, 9766371523208968192, -18770237329079734272, -39095900595060047872, -55455049673119899648, -41509935467185324032, -91590064811635736576],
 [-31853856026062938112, -61211672780649766912, -7715187704059580416, -8152426558961184768, -3387936671507464192, 3087011027938903552, 38659635015354703872, -92174473836295847936, -39240488537821782016, -72647882988352233472, -7829985480575472640],
 [7358053734570372096, -3199010274806318080, -63215289330116403200, 64428659572261380096, 5544741215986746368, -21067289733768155136, -12513416015581792256, -10713399880049020928, -57951318000908828672, -31862296507303358464, -91907417089497300992],
 [27485259945356218368, -46411818795574558720, 20880273148494864384, 96766144439662002176, -37435200145050632192, 16090965745934028800, -15672353815254169600, -14144239199689834496, 42809662129568759808, -71094401118950793216, -8707808047623111680],
 [-14909731444623017984, 51180390227442663424, -13906164552536131584, 1238066797803740160, -647407812010749440, 56988614059090919424, 38437846959331491840, 69038650221234192384, 63867895504235053056, -25347111253134700544, 67289792152222367744],
 [-60772361095269171200, 27756128144964337664, 68687888375366230016, -42912377894579347456, 47636638159610986496, 71411585124271472640, 25520740626952318976, -8223270148556112896, -30200529744873635840, 40610563847293902848, 10286239456255574016],
 [7508513033373948928, -17967828624645658624, -14834196385211703296, 23756023591416397824, 77554648278788718592, 41442749627383259136, -67376634269709336576, -21571089655615520768, -41985105724888014848, 35861871051346694144, -69487252006460194816],
 [9357559469200775168, 32547750997196316672, 38362431184170622976, -61158760289947418624, -79603959979732713472, -2501222279328209920, 16048201438276440064, 12644382594166982656, -47089579171795574784, -77449250523691909120, -21042689252716769280],
 [93321820602176454656, -580574029699450752, 21148482560743952384, -84978888647200309248, -17188523252925059072, -33918169901265973248, -30039444875466964992, -7173707192869792768, 19933696630522429440, -50056124409148162048, -9869986286280912896],
 [52535966812821528576, -17177795979607625728, -27454225134345383936, -8943837665381676032, -69586731863716438016, 27357991345115086848, 25973795270194208768, -39599663758551212032, 19336396575917010944, -90084677223362691072, 39579674505156255744],
 [-8938487997488871424, -11014819288570288128, 60236001699361636352, 16103712054042963968, -61363801279259844608, -38125213228199256064, -78342584722939133952, 31049287252090466304, -212258321753973216, -70416590374531227648, -20246855657468612608],
 [717537778391755776, -20144178780716675072, 66136243591011254272, 26734697043503693824, -26998812299846668288, -44277686702132273152, 54216929487464660992, -8464621112813794304, -37140296176967770112, -27319018085074956288, -91960409596833497088],
 [25892076621800325120, 5843210344848989184, 24214033700153954304, -45385123721439805440, 36658681001076502528, 26426718585685061632, -15197702249520101376, -36140946834431655936, -58792581530639835136, -98996804556043649024, 39888549305906626560],
 [-4914300267029561344, 815320371242456960, -40187051448431747072, -60323992876805513216, -52161481223739269120, -62161281177542606848, -7875787212028263424, -48035337989108506624, 2896705347979213824, 79315030897516806144, -35295062241422336000],
 [-29530671219815673856, 3546917651521142272, 15635255944413958144, -20750196044929040384, 6058201915730194432, -54181319206860537856, -35387689517422886912, 75853884004200480768, 66313886871505485824, -33646827118243311616, 68631919939535020032],
 [26619192492978860032, 100486992986786594816, 12393740916736927744, -4339404254991491584, 16711581077113974784, -36763656399384272896, -6618080242040207360, -16747814105893869568, 13408584990599368704, 2888398629546456576, 93038456757289189376],
 [-29956023979475156992, 15671731048174059520, 6107704417630050304, 51922615127036690432, -3790545858819576832, 3456810453302865920, -41167616746357751808, -82367912673495367680, -3054305195206257664, 89968946556589293568, -35951903057307496448],
 [9208704705326521344, -25276808500840083456, 14969184316815216640, -29632071068732915712, -100890384937276375040, -12206263678558449664, 32568859383693320192, 38715996912428998656, -26000951492376485888, 79307437933849673728, 9948729731386003456],
 [-37931474088588017664, -52331705558709116928, 60274786764964421632, -37627831680289882112, 57297754188910993408, 31136176182318260224, 57825771368225587200, 12682448190221729792, -22519082916927180800, -64952713295103729664, -8284999976383918080],
 [7274843253620933632, -24402632952091639808, 6723765141422601216, 60105568953857859584, -31461625571909283840, 33929131916769382400, 39233490239644524544, 29556121482381201408, -101634021861239783424, 50079485076411392000, 8971679425238173696],
 [-34052011412816691200, 20089244379449765888, 47096640001119019008, -17150311594779789312, 37285823662286913536, -23438448068832579584, -88428650136867717120, -36541990994443993088, 9628731120765816832, -41748790210423758848, 69083405118769963008],
 [-20143935093626318848, 30634598633848598528, -61055516622318379008, 2780952565325041152, -34503206361770672128, -17890720511777828864, -3487449531893386240, 76834906218785570816, -58938773478063906816, -73150326865348952064, -21148081096675794944],
 [-26428695204416630784, 32915461257469988864, -82244714163024609280, 33366380247519965184, 9926072272233369600, -1400839972107487744, 76993290680244715520, -224811421743783616, -77974740264726986752, -2973846076629042688, -1117645378560280320],
 [39462536809703530496, -33318353975964962816, -108570479004643573760, -8014350379918714880, 6347204101345857536, -40842858461386858496, 14063200635364583424, 17321563287174617088, 26495880860940685312, 63102126197365047296, 19202654745029857280],
 [49971686800433324032, 67256564052236713984, -34858921968850182144, -24410828141967831040, -47117518828954116096, -65394493115805556736, 17436571601392406528, -56019126077630447616, 51097580929105387520, -19913654908880920576, -479965541855362880],
 [53754997544357421056, -76758411729423319040, 40822820653467123712, -72921988664467013632, -14073088651417196544, -33274546329487966208, -18141368421976295424, -20133076839135305728, -6837564071282418688, 58838110439601766400, 21543584252597358592],
 [-51238749171354386432, 60121356539259027456, -70491146269934608384, 21548557383786094592, 27725101057861537792, -55363572705325080576, 12737949855153811456, -17461363684130643968, 806275184937554944, 75626296914164285440, 9153942439803332608],
 [25938347636970532864, -14688551252165165056, 58885141858065473536, -11249622273652613120, 22795411706156412928, -86188874476585598976, 20225733141961314304, -39498017735031496704, 52712108337504141312, -14931718992569190400, 66156307121797308416],
 [-27214669653325746176, 63738595450390781952, 12003233151353792512, 6112426906281912320, 85778782280006647808, -89983901339938865152, -5807694362994114560, 9912207791676383232, 33246357682081697792, -10041965962483126272, -375197826351693056],
 [-36433748907894149120, -1679815793501424384, -57287812217820413952, 33889844550053847040, -5849608298947073024, 86790127009617395712, -1573931840929284096, 34434424073382912000, -45535309765041168384, -24033947416894132224, 67907074004753317888],
 [16751156017262729216, 34131634016490143744, -59309835026072993792, 4594340105250201088, -21304803395996188672, -35297844002684375040, -4981200568661742592, 59916687194741907456, -68930811602434277376, 73383015260988096512, -36640952541175521280],
 [-15506464044336191488, -24962252598476070912, 52416704831584919552, 7437935102299465728, 18996368921777356800, 27403056521109999616, -6914686655309860864, -73432857342256406528, 56023004371758317568, -12152473925689456640, -90958378216362196992],
 [-71145190438414098432, -12414744822565095424, -21207436989498454016, 59963884417438736384, 72245938597460705280, -52758646685792313344, -42414513794979414016, -38862819168882794496, -34636465491881590784, 2173581848931424512, 464020834156646656],
 [-89202442689601175552, -12062596831798345728, 19154191257523044352, 17377312125762641920, 3506969440980722688, 100506246256930897920, 17446858420433690624, 5742336358256827392, 29857324170554843136, -34703765063305105408, -21439511567862726656],
 [-50967279001000116224, 38900061795189350400, -41180632494336589824, 55609977785149775872, -52965446404623679488, 69856899137497047040, -39333469631783165952, -6922927894661629952, 52985795187472408576, 23953096452287406080, 14739744551664064512],
 [64889200905989390336, 8945005175424677888, -84889076474617430016, 47868776598005702656, 13636722690286340096, -9513639330496735232, 64564490379938635776, -51458720427605327872, 18788949399669387264, -14936523841526609920, -17908312715289704448],
 [59011979625794994176, 58945719152347062272, -54546437023987965952, 26762364982239555584, 57097022321820393472, 59152619536386039808, 8899676373527104512, 1651003576754119424, -48922095483930378240, -41953413195228635136, -11620208779351009280],
 [-41521421180136701952, 91335221615348006912, -32236773194138959872, -38975724684508127232, -50996293489025572864, -57281573620128194560, 29733178212953280512, -4732487453589593088, -45698262220195856384, -7819588589601536000, 14692433839533338624],
 [62737181237455355904, 38216161481214189568, 84173738107161919488, -18418426971513735168, 16986372072893360128, 25897171790238502912, 70128032216041897984, 22303650784492036096, -29257846908320616448, -39902577733229895680, -11591140440551165952],
 [71506379184532889600, -11465019020565987328, -3916868776810224128, -61308722154197835776, 15770732215368003584, -22298863942466781184, 45310573229282336768, 79942813171417923584, -6680442305235513344, 55793733680471269376, 20296194354108231680],
 [44091666931136192512, 14697058285003640832, 32378192691382988800, 18081782825910108160, 81256512730099613696, 16680975255834816512, -23725196470386716672, 8385323534597898240, 79911181433522225152, -6471316702552075264, 65581593324425551872],
 [8379892578712437760, 114055284573602693120, -51460091273420218368, -7170426997326289920, -32574397623167832064, 41848385496809250816, -35689289544724037632, 38905118624862674944, -13626808125700435968, -4489465287596027904, 13932958625658736640],
 [106033024315800535040, 25200238775895117824, -48729153806420647936, -9019210288632305664, -3498554800847045120, -3983843306803578880, 407382635575684608, 31544477118134697984, 40676592534799106048, -22928936038936780800, 65119032851591888896],
 [-62724467360358645760, -38161425950791335936, -84180460865664172032, 18425988399478306816, -17018504047638476800, -25878980549491945472, -70054826779746459648, -22361243049181339648, 29286844673775058944, 40030618650780524544, 11589928644860248064],
 [-24792002233236733952, -57249652547972276224, -86856777437055418368, -40907594732211109888, -9844564614992957440, -30662089887160483840, -24317567226898145280, 64097660597098233856, 35179817127576580096, -40211352695897841664, -8767468305758889984],
 [30747153759393669120, -35488832870795079680, 108014604179100254208, -347256694258072576, 48985050354959417344, 16819066512174714880, -2015490268588259072, -10801976503368996864, -25665970140137996288, -8713071588621085696, 66091835828351483904],
 [23927297377099763712, -54203183050957758464, 59177282000608837632, 15948410349396647936, 36790946796433039360, -44397033189013233664, -1938586268907908096, 53189032012772515840, 53468340001640947712, 71346275040195772416, 20114981050837512192],
 [20583419339487232000, -55092176239206309888, -56723497689958727680, -27733540603256606720, 28767084656634347520, 13557758654731339776, -32265253959635832832, -85996325320060649472, -13255072838828038144, 65445882688853082112, 20637560562884141056],
 [61875075769092718592, -20189459124386439168, -17161499208519454720, 1204955030417256704, 7416992516577055744, 55239938354478342144, 26396018520429359104, 26196212882363715584, 81725131243442290688, 75332292775055015936, 20063845794201948160],
 [-9417438357796257792, 26118049010143977472, -31836418855136350208, -20761233833227837440, 24688621333209407488, -44657875827885457408, -97243287526940950528, -20105272500537933824, -55633066473263874048, 62405398615992418304, 10432629008169588736],
 [60757198265934217216, -27800988814825193472, -68682524195964076032, 42864712364535717888, -47676234289276542976, -71502645879637942272, -25512199976196673536, 8276403261552057344, 30155950295656747008, -40481835372831932416, -10289968462199152640],
 [-37003854560971522048, 41698324829189963776, 53273077562236862464, 3615564205119427584, 28552617068823330816, -34469086034698813440, -91553065620487438336, -30109797423935389696, -3137735994966443, 1868310156707774976, -67257108827865653248],
 [-100871773144376197120, 16189920358551394304, -21952343684788932608, 23800824825349419008, -71387519904645758976, -29275264524746616832, 26091343131896590336, -50556875290195623936, 20924524759982063616, 20635094781949902848, 15454477637735516160],
 [78908630274440888320, -44579376363084521472, -5099828430389444608, 12261866125646804992, 51303987085054525440, -16683315115173330944, 30304001634194157568, -18800823425177993216, -1272116397227034112, 48909194000410755072, 82533956859367522304],
 [-65074522101956960256, -8856504812964619264, 84826648393377595392, -47954104560941735936, -13778317717282271232, 9495108516817823744, -64686956943710543872, 51550070286366162944, -18781510924199976960, 15072267750458789888, 16372363267892355072],
 [-17009386381640912896, 80412886732264128512, -23298024071211798528, 65779523629462732800, 53182658349095763968, 3435691457132481024, -79038990524805857280, 3661231014442902528, 20026592997842575360, 23549936232728977408, 9815628683319162880],
 [47243834331302305792, 4230159775481385984, 95767067007783157760, 73379707525404778496, -44744444190957486080, 19119582780574777344, -9508027305754449920, 40308220929784553472, -29344666710191575040, -5251184841674220544, -906285328879898112],
 [-78463385073031512064, -46908283139906699264, 25290958602387017728, -37018687806384414720, -40389967890182119424, -42371128499825565696, 19253423106718400512, 62741853820557279232, -18926669398590115840, -53826479952907698176, -20536808598803828736],
 [29368360280831823872, -4573875839073743872, 51283657722569678848, 11689163545051895808, -44699659235856523264, -51877807103302991872, -76484501925235343360, 21918359171685089280, -10694178584701626368, 75679052359945584640, -35590112847460982784],
 [37522858015323521024, 49335594381594042368, 54400568096880934912, -31878501063379996672, -7001184950320272384, 36076538217608351744, -51012571927171276800, 56343268342382706688, -947759003653232896, -47121761941988327424, 67909548803959250944],
 [37960219891024625664, 52314745456929644544, -60274904180146618368, 37644963644382380032, -57265517882417192960, -31119002454235566080, -57801376168859516928, -12694756203046305792, 22527955911043174400, 64871670455756464128, 9182757506431917056],
 [-29860333714897940480, 63087988964059865088, 8882588725848359936, -45402057009612259328, 23899957141453942784, 72506637999131222016, 21106676389780951040, 51578277398014361600, -36715294848328503296, -63268986193695121408, -21929020863264096256],
 [-83595772290598764544, -55437041157251981312, -4512815561980313600, -40901841089406443520, -4426132972275580416, -3527350867276995584, 69775572654166843392, -26126112180683804672, 61706759849988579328, -23472544111110737920, -9016929890607316992],
 [-55351787080671830016, -18432303842616891392, -23838713408179511296, 25622762114020470784, 24328699834484428800, 29140496931212926976, 75788430801376821248, 93567006447036661760, -13967306464851124224, -30083285444885553152, -10482188740410923008],
 [-90254292607046369280, 92117657606560874496, 37986492402416730112, 30138816515658612736, 22297880826792738816, -14026458862978777088, -388595716535699200, -34736277800431722496, -25129291513247059968, 7038641679942300672, 14612618920029511680],
 [66065180336674603008, -66244636571083415552, -5315682715341206528, 17747946779776567296, 57129643534372839424, -2935307436022376448, 39398246242780766208, -17433451623599644672, 11857993076264400896, -82518374004481638400, 24564449254806110208],
 [-37841687914588610560, 63111818386276859904, 56319094643830095872, 6510404866614461440, 65674211741787881472, 25805754744019918848, 60848786051196502016, -95217783724084512, 52509033690079862784, -39927886892099952640, -10756070787514810368],
 [37902419199552274432, -63151880156776947712, -56347462239077498880, -6428731198105591808, -65648026137207611392, -25746484076949667840, -60853100684812247040, 74836779378193520, -52454355911049838592, 39946145368532942848, 10684045275137105920],
 [55451234991312846848, 18413060641613477888, 23851869389930582016, -25659947483991945216, -24275011583409610752, -29042334867772059648, -75778507951605301248, -93551290118046662656, 13979268333030250496, 30088522236840394752, 10447390813900658688],
 [16685671252052008960, 67065631180552200192, -8136295847977519104, 35813077984548761600, 33020348778689531904, -12906810099098652672, -45636754195551977472, -88962622511639707648, -62814465131291230208, -21288426393331277824, -490164483363749120],
 [-51779069394586378240, -49016662394944651264, -27477680409174351872, 11741072867959136256, 429501307644460864, -24589952547005628416, -75785209131505025024, 41202169965467181056, -43270366332861538304, -32776532310168547328, 69346938699593842688],
 [7840366284323213312, -16673016561144922112, -607330404505168768, 6184831807323745280, 12032722463636803584, 3788769060401526272, 11039393242325579776, -2124136171720182016, 5777557739171574784, -56321349596171304960, 133824488533437300736],
 [-24768216066150998016, 27220713700839641088, 11648671239115485184, -11931319524633812992, 17849932369090609152, -53904911996553125888, -64262457618295988224, -17108702539320320000, 78832422328627167232, 78601756937667870720, 10299222576233904128],
 [-70233711272816812032, -17132649704755421184, -66688369938466414592, -40094984763427209216, -30405621887372709888, -38961736153043075072, 65835838053787369472, 44139951826284503040, -9786003115298945024, 39547634222619918336, 10574981521703522304],
 [-54901253822271709184, 28985727926316769280, 80323543132870000640, 16921372901351325696, -81790376214722002944, -45967955445419474944, 12195717638279802880, 11127840507546836992, -45003628350310858752, 7005821145524682752, 5987981221715336192],
 [-43873872256586760192, -58323248779519156224, 55820414187154915328, -51816394046095532032, 19842971054042861568, -25149636011135664128, 50633549250683904000, 43620020522439237632, -40356308163236446208, 58497819402025189376, 10361441057478803456],
 [-14450023833596919808, 46174796346691821568, -11348399519742511104, 57169853557979422720, -89378480206552383488, -41762808382381907968, 13103729563127543808, -18823670854996004864, -3548615674109921792, -72878555850326736896, -21387341396732960768],
 [28481501209590509568, 4308190563395570688, 23336998480246788096, 38639796163123027968, -59611932939288182784, 67869233431496736768, -6874198030397076480, -95939175759654993920, 38400130083137404928, 4969014205311097856, -95855871801507504],
 [-37652915092617101312, -33801100827680899072, -24617859554481651712, 44283525917168238592, -37634656473064235008, -8596773930973555712, 18326579265833506816, 38725922007818371072, 53812851044848377856, 39960194785179459584, 93980172570875969536],
 [-7912199639609114624, -33424252603679772672, 15318179666622853120, -72582788692480942080, 59672811060820164608, 3521758251010411520, 45704513608236621824, -70188449834612850688, -6582398923883951104, -4604463665589934592, 66650452349109551104],
 [10743062867709335552, 97781438885084266496, -64338697274451288064, -8646294775195708416, 56390828220342886400, 9631208917206431744, 22505212041919266816, -17082525271060918272, 61488345288656003072, -5875525933430780928, -948983565348032896],
 [36340159792511361024, 5512010303034705920, -9730751382758320128, 8269434271130028032, 69411131362548580352, -70295145252960755712, -69432339448539824128, 13898149849144680448, -52587566490535436288, -51228059401653387264, -10046083457261901824],
 [-26076854551665479680, -40863704033384390656, -23787937685971718144, -56524031925769060352, -8350305272906653696, -31833232331223900160, -51577026086021931008, 68938594247452041216, 7195357822708967424, 83674093704880422912, 10884202349227118592],
 [-9476415663828152320, -29735111344547323904, 99679321955010510848, 53398103079416881152, -30224271080875892736, -31745625844496093184, -35187302930556858368, -68943717758534041600, 10159650401820522496, 802010965216346752, 481661855860713088],
 [-49872732922871013376, -45436423189796618240, -89412664078948384768, 9392622554419802112, 9838176533207465984, 59763402947920429056, 31535557360543363072, 23105454090732306432, 56419529163640209408, 27628863749722529792, 377843851866836992],
 [-69990605472344547328, -18822430651210399744, -25225130385945157632, 50600361341878304768, -52664433868061868032, -5350617133264600064, 16049389267389020160, 48235433518846935040, 75083340789946155008, -43384521769499828224, -20549027415270547456],
 [24696821319050895360, 22537952321664442368, -61111097329887453184, 6788037299093618688, 41501944451346661376, -52670484435892649984, -45517101531224621056, -62972536120827068416, 30147532389939388416, 24017624698632249344, -69712820707318317056],
 [-1003803344548086144, -27171814549541765120, -977380379819870080, 6621767694642054144, 12774674034793043968, 10519075067548884992, 12605662437818959872, -1550314550350002432, 12272290157715021824, -116363034176451919872, -82872426113416708096],
 [11910528048005150720, -42858999626970800128, 2055192742348670208, -30619054965581352960, -11001784031357220864, -43886010294166405120, 91804871477526396928, -17422787512998957056, 49646687134538309632, 72107024489958408192, 20362735178968854528],
 [-2628985795789189632, -41207726945455652864, 6354654570678905856, 82771646984122859520, -34906029985855381504, 78884079401711960064, 22027696511024005120, 63413329627997954048, 27073474392732745728, 22376568473058353152, -522919299802364672],
 [48882104299961196544, -61578823557472444416, -67683853333059371008, 56442707495128457216, -22113037193236168704, 34606443499410972672, -27136281124384985088, -22959762796068880384, 21918050316413898752, -1905166789844174848, 66299511486608875520],
 [-11807395900163661824, 42833147071626641408, -2061977709939641344, 30612214510636171264, 11051909516062816256, 43841153765063835648, -91799584129399209984, 17392499240078852096, -49669925881103286272, -71799582341400469504, -21598223598989320192],
 [54136512621807894528, -88789857375068094464, -20518341049663033344, 56496665600538959872, -42756971130511679488, -19138599750421831680, 42078753379407872000, -34406752676893241344, -42751653601098727424, -9001960129787187200, -17122070546279075840],
 [-51798226167611572224, -23449548594100117504, 22071199743094104064, 4927107297901444096, 24108578578712436736, 56861050861243351040, -49093253465918062592, 53391199498988519424, 77237378095106998272, 53564275094405234688, 11126776163885651968],
 [-61765854390989561856, 20156455900322754560, 17155558845708081152, -1208879413653470720, -7367573101925479424, -55284050836365754368, -26387026581584859136, -26226686662924218368, -81745191821394296832, -75057271667556286464, -21222660520484167680],
 [17413158046176305152, -48560655782862536704, 13628327699779508224, 6052539500530648064, 21692835720676642816, -63350687408348569600, -19093564545741606912, -62907214348429418496, -72039101289376849920, -21048829123828891648, 66752495882309754880],
 [25068679629701230592, 54411553903763283968, 85052203773712498688, -73571924223207800832, 35944659678978564096, 17154535990874580992, -9731060552681398272, -54558538763449344000, 28113632934503858176, -9711008727411748864, 76042148472325840],
 [-42252251518938710016, -15398771646351171584, -32704159975085608960, -18718388428374036480, -79402377950892818432, -17207345396408729600, 24656632620554149888, -8696186785951845376, -80147926766018478080, 8384862539847607296, -67508197797899902976],
 [47696037945207521280, 45383401812529930240, 15190566611098228736, 32192872509964369920, -53581980927068741632, 66972174211656671232, -504227543993305088, 12719695210038730752, -10466385590462255104, 84707926077990584320, -37054666030914469888],
 [-76630277484965756928, 43153408327490895872, 4926917292296115200, -11621275404224071680, -50070171028657881088, 16330288693847949312, -29017475845481865216, 18540459213340553216, 1208959845621473024, -49353709433624584192, -86530998737429348352],
 [-35240609175724847104, -15595370581595529216, -38642274757658263552, -26800342845634789376, 92544443679254364160, -19611479125994360832, -84220095448968265728, 30305769191833579520, 33178949943825797120, 4342170880405459456, 1107306964265630592],
 [33187032211272269824, -21628594048719462400, 97548737015958159360, -45205335804699541504, -42858671053894164480, -46857022045123100672, 14101379255768340480, 65857444536049385472, 6395417812228494336, -6340782190841409536, 113277702884359280],
 [109782029024511000576, -15820348992276619264, 8945706097544662016, 49581459503996534784, -9674016550077093888, 71666447062224601088, 37832507184444645376, -4731887981562883072, 3371968842961107968, -13350225831239204864, -8024528755388332032],
 [61049858133684420608, 75863265329257218048, 20672883506198454272, 7459831762252612608, -85853934709488320512, -43539779252795400192, 6482533682488125440, 47370792681727000576, -11535753205558816768, 6531966036695101440, 9185214692686281728],
 [-86551315147198644224, 29149754092366106624, 12799617403346350080, -65412451231089696768, 38398485037787201536, 21611525703626371072, -4587140794561542144, -57680385895207313408, 2763285323981484032, -56935289140030242816, -21349023736176021504],
 [-14057174122767865856, -25858942186110222336, 1782308677493372672, -118585008673950662656, 1886522160744721408, -65976502790387458048, 23608894287296356352, 25549092250563215360, 35740522351783690240, -1089229953641335936, 1017983661492154496],
 [63971698304883834880, -20012234894115262464, 11290003674731294720, -54396238992363610112, 1622275849815774208, 78075455591471808512, -29759653094970626048, -27579073913716211712, 16275991871943112704, 36739721858743169024, -69565960535944536064],
 [28684175432459960320, -13803638050371471360, 31724609109703946240, 36569721592982114304, -64385924483580272640, 22729373030826336256, -35771470524715544576, -57345975310419599360, -74803578399522439168, -59156056979060506624, -9777970252336111616],
 [-64414727086058037248, -21069479001847427072, 23510179420247781376, 81574374020016488448, -42266208562558369792, 23620803803356966912, -4340659232231512576, 37194039685703311360, -54654923616883081216, -52868165106263392256, -21188870009436811264],
 [-103971478692154507264, -25554853248839192576, 48294314728891138048, 9508688710913021952, 5157987983953737728, 4006549522870359040, 325223441574698816, -32022788031784112128, -41384718000435380224, 24791425611698909184, -67057176426014801920],
 [49962465453660241920, 22677552028622213120, -19215846218195935232, 31820910779315695616, 18411035141679030272, 99123077696108953600, -65415495110111780864, 43618972715310301184, 32075450081342722048, 3257044187315528192, -371067485248395840],
 [38201923018845200384, 65059781962774831104, 66749932910534860800, -30709822694586134528, -29260400357194518528, 24989662645863112704, -50202955567972261888, 44396581418452238336, 7900234525169259520, -2208093359965817344, -68175697565945389056],
 [26428665573875761152, -32915384452087078912, 82244718786723495936, -33366406394003881984, -9926123438873606144, 1400818185753264128, -76993338723111862272, 224819808632679200, 77974711107616489472, 2974120085276719616, 1117053691265366400],
 [-7759353174903952384, -24939215156043161600, 5393143192298496000, 85182767758290927616, -108454964238900494336, -83929933098472240, -20335197905051787264, -15672039109452197888, -12357491004337862656, 33637466606556856320, 10242236698202081280],
 [30158885418606661632, -44053286093564084224, 2664775566515555328, 25824585826980261888, -101319440204564725760, -4874132658076752896, 25507399699555880960, 70711169434462969856, -6476106092576146432, -46603118984844222464, -9996856088594126848],
 [-24803906086678958080, 8973984451904638976, -4360167197566650368, -64180865645897048064, 45782771375297527808, 76918752846286585856, 21805062822941282304, -31452252682832183296, 84531628365550321664, 17917753488074358784, 454987729384579200],
 [40028009600899153920, 74330223129870106624, 7560774616653791232, -27433451264144674816, -33422574218333044736, 39864285332144627712, -39616990050813911040, -17310142813286342656, 92486698139224948736, 24664106669785489408, 10178612424497086464],
 [-26730489478904274944, -20918073491431804928, 61433413829285879808, -7554829505431570432, -43252680481624080384, 53354495192276951040, 44909191325511499776, 62902501998081236992, -30226506641457770496, -23883730856106917888, 67968491085484998656],
 [-23110219657606340608, 13798961162596071424, -59093069344319266816, 10701042225853054976, -21794887966181130240, 85713271971105177600, -19872208530820108288, 39114622822059835392, -52783361464879538176, 16865970262941308928, -68043295243546329088],
 [59531362625333886976, -15372914430270058496, -2908311520104891392, 3275822043814204416, 29075040580312715264, -20692899684272975872, 12411755254203992064, -13552127490486853632, -9967377585713539072, 122229854036312784896, -36439275776235569152],
 [-90472580845449543680, 23609695617732276224, -27252911206980685824, 28428955680921706496, 36983928770562613248, 3738690093579274240, -21978117816352407552, -12468824404748787712, 64510792274705342464, 27411991382399008768, -67461704347070889984],
 [70277474318694203392, 17188297479562569728, 66689317379178954752, 40165809674107830272, 30335270525527052288, 38908554880731742208, -65830580790581559296, -44184952342489645056, 9771364528163778560, -39457941978914930688, -10510222897304465408],
 [24807720137212780544, -27248121318235774976, -11654704093653958656, 11937442602011086848, -17817308829818478592, 53911157442489393152, 64296357000197963776, 17100445094774431744, -78821766984187789312, -78691060362467131392, -9294280909903785984],
 [12991841948315938816, 39323041660183977984, 59599122037461630976, -11615306269556590592, 25811558612514795520, 51356826347835629568, -84929362086923059200, -11109498068829990912, -460455428086934528, 75076350830399029248, 9958066949823586304],
 [41364304183374880768, -91186703619219865600, 32279287401438912512, 38950129816454053888, 50784645169411997696, 57307308906416603136, -29883913321609482240, 4721410404567628800, 45600893461453676544, 7992270955580542976, -16465727659350802432],
 [90035093358529363968, -91994081569321500672, -37892253589800296448, -30275817895576866816, -22473800933977980928, 13997093458544465920, 238940554485460192, 34737688867938566144, 25085821025192648704, -6856616135778610176, -16502372441413437440],
 [21464515195148644352, 43011936988727910400, -28782486174046035968, -29603533563159216128, -69087839519833628672, -8631783166203142144, -28715067572485419008, 50342007497367527424, 64254293018892853248, -70570097888565788672, -21148420153931300864],
 [-21382594729390612480, -43040410759534682112, 28779021839357825024, 29601917074146672640, 69128251900304900096, 8598806349179473920, 28725351766257876992, -50364680994606514176, -64266767463372619776, 70761381665753694208, 20299788966642937856],
 [50098908757813157888, 42639423507735592960, -48638347065229041664, -1657737363701922304, 100020169171826933760, -25589721327377293312, -12412492882821632000, 33149527952922042368, -19546221675827830784, 49556991231647023104, 18216313160575213568],
 [6101082489008809984, 62665054562993438720, -68900867055657459712, -59294766852437516288, 5588668103008280576, -69346782349756440576, -43239068302339792896, 40679140175763963904, 30137450211656355840, 12918431688757014528, 10534214058582724608],
 [-16449829922206492672, -33870040888165855232, 59367779395217817600, -5469245037529663488, 21096379982899351552, 35125709745726332928, 5346627545408584704, -58782782362636517376, 68776927238637690880, -72740730267380924416, 40267788801443897344],
 [60708896008356814848, -12154334057723162624, -13074286494573539328, -22583831923247009792, -25484733371967377408, -43772357366994771968, -63610737003840176128, 100620989358946811904, -16716652426160844800, -15745718915745837056, 181001547698170816],
 [-47820541178265051136, -39471463578764992512, -41328306728887410688, 84193421385341665280, 37675471144302026752, 42140282901449801728, -60120892888671297536, 27091935257768394752, 25032036459008745472, -29619166903837384704, -9437826105484363776],
 [-1684146621936307712, 43457210234032873472, -27429486031579897856, -31181979728725774336, -128245316633434144768, 23366783991351365632, -9815491726829398016, -30579901657308033024, 7834197471736433664, 6063364149709200384, 6287473058007268352],
 [78555955388660154368, 46885854520214069248, -25296819076295483392, 37014696713112854528, 40430315671982039040, 42330772234040950784, -19247380568891383808, -62765958558679203840, 18908008406127157248, 54077027561045090304, 19502911734854234112],
 [48113644188245598208, -79896834557518151680, 47580871678550507520, 27068902995873161216, 17882932447702216704, 9076045098337236992, 61257742258112266240, 70361352381103202304, -25939160519437762560, -1319498968165618432, -17406717251379902464],
 [-19094415052267851776, 104543795543495393280, 59178747307250696192, -29793277083782979584, -49933135271798882304, 38748000588504719360, 42034353565352624128, 4168494973750313472, 9486694879406546944, 4851132033383797760, 14205156755314642944],
 [-48705231340143976448, 789698899305356416, 70199108889578151936, 69163743398879182848, 73283744036366573568, 43280281549887315968, -30117803099186458624, -30007170430807498752, 20550214287199387648, 14839009849439113216, 15378004860174882],
 [-59698933206960701440, 15142476195462225920, 3219674498397265920, -3342960065212910080, -29372581974408683520, 20543674428077883392, -12518303849307629568, 13498412317780090880, 9664619395714924544, -122862396391976697856, 33926288821939875840],
 [-15084847534183299072, 58422161816104345600, 49985725205485617152, -72366045445160042496, -60902065316114776064, -55874826458317373440, -47253808499331620864, -4481346655845801984, 32295607406507089920, -4837664066198289408, 15858142401453004800],
 [-32236374751261134848, 8271565983899476992, 64304183117933297664, 50511531620101595136, 8584003854003835904, 1095066797535612928, -64260520928639033344, -10054080848254289920, -95106175808551567360, 22956601183968567296, 10317578636011626496],
 [12003477291354630144, -11365087978222225408, 57124930893262258176, 46579142398022967296, 47799738845343588352, -492627516797424064, -93728440827995078656, 70613800981634588672, 3834051388958571520, -906730289841013888, 198041566489406400],
 [-26004958885964771328, 77016843441193451520, 31523268776908288000, -65577306151997710336, -11308746491315290112, 24620246470540705792, -53677533829745147904, -24860388885620686848, -74708258597051301888, -4623117990989549568, 6329445984821189632],
 [35689096507014942720, -19204376216371142656, -12864584075494924288, 35428742416635281408, 79929680686978056192, 39536411004787335168, -63152044405668929536, -14989607855659966464, -44791590203496669184, -12577829920419371008, 66224009470316634112],
 [56491520967800242176, 42977456179905601536, -13498083394616453120, 47412118994492628992, 26704548394605256704, -63220934829435592704, 34894945584509952000, 1610169533924766976, 70109172913625899008, 55484899968647421952, 19633412566541709312],
 [-9094519223292108800, 45488227619277824000, 10477359116609720320, 9011587632411295744, 48896655465656647680, -78545812733458120704, 20689377611713802240, -6108573397042640896, -97551497861913985024, 28479028589341085696, 9064003105124729856],
 [-30112843120331554816, 43964122143230181376, -2601369283835485696, -25860061564565086208, 101340649336752930816, 4829104740998805504, -25423797558104047616, -70764539318304915456, 6436152552172343296, 46615363818927579136, 10063848598029803520],
 [-48346217777452761088, -6357157122649085952, -41700141855450300416, -63841375918965481472, -63317740595163201536, -50543481672860909568, -9458822211450755072, -42921981947761238016, 16815056014736033792, -66467843112514994176, -19733530105639591936],
 [-3827701097199844864, -14050046793206685696, -55426551848665366528, 74967007815872462848, 10711371102551726080, -9639513852731854848, -12000158525848752128, -28338234324611293184, -44145323894855393280, -90639293414096470016, 39431678911471173632],
 [-33768869684626841600, -5955738222879106048, -40293790761508896768, -88704243094329262080, 13742157953666363392, -16041697736719169536, 50277086415259713536, 2460393245969906176, -72166607906271641600, -53189724530718949376, -9645919931752128512],
 [-26368737613462007808, -6884472966015893504, -24326613456722919424, 45415461943465050112, -37089586217321906176, -26024045143654612992, 15245624495189630976, 36455249163958792192, 58517839642912219136, 100137478241515274240, -36261028549562888192],
 [-14368318380552656896, -29371163747914162176, 14836449407087925248, -47293797469014269952, -5776357303989607424, 44007783671853932544, -29164910514077868032, -13508636670631317504, -75822497438523965440, 22918217835265748992, 94734955090157305856],
 [-53668901629851885568, 76719224387519119360, -40826963857806245888, 72925189172671717376, 14123062380030357504, 33245872232201891840, 18160071890739841024, 20109249023127937024, 6828223291274508288, -58696923559291355136, -22272952408343461888],
 [-22967006934998904832, -38043226083411836928, 45715402268250284032, 36327867741677584384, -26974454925257760768, 72789327809773469696, -70611023385121619968, 1962727868419701248, 12411126588163389440, -21771576513636683776, 68920062772604813312],
 [41946858583009255424, -12076726816328726528, -85503575431467859968, -57323622741201084416, -40351317639945052160, 4978840437872674816, -60976155988510449664, -35625716903187599360, 51029038972721405952, -5528269135717579776, 1000251027183422592],
 [28858464941309018112, -73352231094305169408, -20941434913112002560, -30682390594332798976, -28832447751624691712, -40645006484117782528, -25017094209199357952, 8421810893750818816, -55972158909091348480, -88393523273611345920, -7638839070329794560],
 [15418648508866109440, -980298901591254912, -43477014127762948096, -8938931796565746688, 6853176138841784320, 9207381396733089792, -32955324887464980480, -2935544511547163136, -134461577031649181696, -16213923821597128704, -27149789521753624],
 [47070507031695220736, -37914670159254749184, -42345875623005970432, -78666434666350444544, 49263989004097839104, -26034821316172619776, -33243456944774320128, 16127490473669605376, -28503688458794635264, -19843486512185757696, 66836632075129569280],
 [47343315073913937920, 26049670938899943424, -28503847843703816192, 58360970462712553472, -82023291096765415424, 13489816275804014592, 39464076268784246784, 7394188338737650688, 78170668132609556480, 2464001735973768192, -1008467248013771520],
 [-1519280692263103488, 98372261310223548416, -35575079024413982720, 52606078692318593024, -40330788872075345920, 16849700116617058304, 39504477878188040192, -63566856692670963712, -4648534017299281920, 21266086363399036928, 8943514177959654400],
 [24726584508993515520, 51795400198114557952, -27693553945478705152, -24739754606701887488, -55940467496822153216, -55789261014951108608, -67163982738646663168, -6720569866191983616, -38735566849217429504, -14560455926765729792, -67782891873388077056],
 [-49971668560891133952, -67256599289581690880, 34858919774812405760, 24410840520971177984, 47117544247431487488, 65394501207984791552, -17436548298117693440, 56019121372655230976, -51097568739559391232, 19913540868058185728, 480283572042047232],
 [-22477838384346136576, 7324610055988571136, 72634859763603128320, 8261791005806575616, 74834436978636390400, -43153652670292385792, 24739548567402479616, -76353454117393268736, -26049920311391985664, -34415505487743275008, -10379644823057948672],
 [34723373781727633408, -30667753658681552896, -62385224931721641984, -78979647382140403712, 28024942959116488704, -19148911696363528192, -35451909266192228352, 10915914903301920768, -39461657074485231616, 25271716653469696000, -69124462629429051392],
 [44141734616696135680, 60144648373142011904, -97314577980611002368, -38984255772319072256, 2448358475752541696, -8855447017101173760, -1035057002272243840, -47812112528239730688, -44757267741532749824, 27583896603580416000, 9313083181233666048],
 [-25848877202222055424, 16992806616527810560, -284841733116344672, -87981622591591415808, 12926204022391828480, -22150538401810714624, -68201016514397855744, 42940727911489970176, -13945383088590749696, -72756292552088076288, -20960315371164909568],
 [2618716443763912704, -67958617238021947392, 26136218571663290368, 103433310057086369792, 19302781386947833856, -12476972831229052928, 3571888937517654528, 13981203025770473472, -26505301285119672320, 161595790966480928, 65819653913433808896],
 [-16685670038106169344, -67065672160490356736, 8136294270054574080, -35813064345952079872, -33020327068837097472, 12906827590959101952, 45636778167986995200, 88962621018686865408, 62814484430778490880, 21288236814977323008, 490422700908109056],
 [-23815705949516292096, 54197084050514665472, -59183118637182394368, -15962407281946931200, -36753326798960742400, 44339611919109070848, 1933075818036238848, -53218431542929186816, -53501519839269978112, -70938261356121423872, -21650028460785504256],
 [-93297967933663428608, -54206605337850544128, -12792314896282316800, 1164161793566649600, -70743312882599763968, 40675965767088275456, -19324882611213062144, 40085773661082697728, -7636341936841805824, 34925582094742556672, 7254319925396925440],
 [17582229656733958144, -17052356725661470720, -56352203381000470528, -10463619789596286976, 95704417791309611008, -22988882913032892416, 25103452611738046464, -58845254431018467328, -59387064431351881728, -17408983847343294464, -16805590841428207616],
 [-40510068104190124032, -12873083806046738432, -51049301578339385344, -51773934965797158912, -69772697314567331840, 57251963296130416640, 47572964645096226816, 35748043502556434432, 9319340652348948480, -49716385512628289536, -20952870331934601216],
 [13851964788472997888, -34469149763863191552, -46703131218038833152, 20742362647066456064, 47355012490217857024, 9509177078885459968, -13684582116747216896, 90299803568639459328, 14578258214435461120, 44569758217034973184, -70095729301011628032],
 [-45118988716537856000, 36634503853007470592, 41291305416348672000, 78963032473665454080, -48370688756626718720, 25943332923671941120, 34202717399499984896, -16840050892706912256, 28252192899003543552, 21977579975375642624, -68649365593741328384],
 [24658878402116780032, -46568602326443802624, -2639451405357859328, 76449903422855757824, 48207175205530673152, 2748492266830771200, -7535174805209808896, -103018342072709988352, 6360942487015439360, -7883467453873404928, -6794687406649353216],
 [94512498472434368512, -14515938123254587392, -33736883096369160192, -15923751866482659328, -64410754429521805312, 4169059809003744256, 27942997027153911808, -53874801034225557504, 7728120941392300032, 53980375743031926784, 20493098043604348928],
 [-24048994179808862208, -3639446467792616448, -82479586741303377920, 15583172264135460864, -58032311273527951360, 34681699318239440896, 18576693069843066880, -27244907413687169024, -32364259961712685056, 85687289183906627584, 9646218386602553344],
 [-24121011401735602176, 44964447868053184512, 9144558753167053824, 34084156774284988416, -28091517915919372288, -112572237762385133568, -38874673063241342976, -51343760800262373376, -8273173759984862208, 12038213968739825664, 10327920887306280960],
 [21018387054506405888, 39644183749768658944, -45504644666084237312, -36539378801356148736, 24875977371603210240, -72402237333869109248, 69581550665019121664, -1966236090533160960, -13186701270279090176, 21656496793448988672, -70778451888940072960],
 [-47343312912546758656, -26049725869636038656, 28503845711676297216, -58360949280307396608, 82023321449795305472, -13489790769684953088, -39464040418481217536, -7394189423461293056, -78170641110030942208, -2464272767498986496, 1009135468670923776],
 [-13758156880276606976, -25938995028746698752, -6347561705449233408, 25227780539818242048, 20958610433879998464, 117313100352590561280, -6870553850048176128, -35104496869597356032, 2334911440545042432, 68217209159696465920, 9951089147607277568],
 [-10643996274200270848, -75961212541645602816, -59932669751584587776, -6372623342137199616, -93670021414244712448, -15262340343884873728, 26465980468374614016, -15831357410905726976, 46028078292622409728, 13592443158773680128, 839843536871480576],
 [-7280866965530322944, 24339455264960192512, -6688303616376130560, -60008349421155147776, 31508924429630222336, -33986137946017042432, -39241919731133325312, -29568192524418260992, 101666507031853924352, -50066197366181527552, -9062948512927073280],
 [25852256514070958080, 11568604709863468, 1589008656570985216, -23452524318298963968, 74706400626166775808, 38643479878710919168, 56506346553176489984, 37877451870784602112, -66968758354933063680, -10544682806268700672, 65262148394577666048],
 [-62762080891221262336, -25069510184902963200, 71981193899104485376, -49421147884516564992, 75170480293031329792, -22696533220432191488, -6508912964093334528, -4458460396717142528, 56290773565345275904, 13750390286504388608, 1033559121956867968],
 [-47060293003070242816, 56327787211250016256, -56116141869190979584, -19606691533020012544, 59051436693152047104, -9584890021443627008, -19230868105492717568, -4231349471087820800, -48484101969504206848, -43697669145727565824, 68046084611023863808],
 [-109769416915010830336, 15762675573077825536, -8957051396893501440, -49612888933974736896, 9733302913641734144, -71737345650327601152, -37833470532599660544, 4695238136016907264, -3375336475941168128, 13392232600739768320, 7312430727095815168],
 [-98309667538151620608, 12806424683231236096, -12693336668225146880, -2760405359792963072, 29053005038288056320, 56358133192851521536, -78782319695925264384, -14328388355569932288, -25177956175500091392, 21487995273005170688, 15893372784985651200],
 [-22802352571138265088, -52697127882890657792, 27859927286640029696, 24806280632184565760, 57260968385016528896, 54422583906794602496, 68392709000947318784, 6700993191389650944, 38904174320159121408, 16612445113835634688, 65855900266578001920],
 [39227717814563078144, -30524452735066902528, 29479908233966329856, -15765489116835876864, -103507897752150081536, -75026787293107847168, -5068935712314784768, -38936961742232862720, -10391448033344055296, -14036361258828703744, 461897678134478592],
 [21090860808943017984, 20811942621805150208, -47550706635046576128, -34287629569792389120, 48503266953982271488, -64374515405846544384, 78838742625020018688, 9142876335933715456, 11328538003718686720, -16772303510885705728, 65253109281312006144],
 [27134722434282094592, -2493216553542883840, -14874131462660200448, 20519810406190571520, -7626750532486002688, 54707398239279808512, 34575011379707113472, -75086425040437198848, -66287914494465130496, 33842597357665476608, -70493839445036457984],
 [2418796595642711552, -36534809194261676032, -333018583959745792, -17898861271309770752, -67914414939510956032, 36136170790728769536, -58422292779263623168, -26907542909636399104, 45657862165496225792, 88659790387299368960, 10787906324444446720],
 [33817804955504455680, 5992453732479644672, 40321068879293472768, 88654183317162459136, -13613602432274579456, 16061602339611097088, -50303969872832757760, -2471685705937179136, 72197431722135863296, 53202352106304192512, 9502734395034365952],
 [47520260878165254144, -64388349400566169600, -47913611998088667136, -48603442929510490112, 962118718517573376, -69994040189160202240, 28192405030933356544, -66078666441145237504, 16652132280151906304, -18439640297328799744, -6117256975724308480],
 [-13485767254347546624, 72213590850275180544, -22550419891151011840, 21854238666861162496, 35699460362029121536, 49906432191307202560, -7931023904592243712, -11439957771161419776, -78483187254901587968, 72108539117857251328, 8783186349442869248],
 [51268978368381878272, -60143813536227409920, 70497597263873294336, -21535085073654398976, -27703484667450335232, 55364246677182177280, -12703967242620428288, 17435051387361810432, -801239088773935872, -75719977817496911872, -8148756102651506688],
 [8025637449835805696, 79333456761457737728, 50645446932323737600, 32912698799485153280, -6766517804424283136, -70373157581691494400, 78412047233755136000, -20393564522094645248, 67861395839362424, -14385846366811598848, -1478292975735890176],
 [24834257594246131712, 57160577542473990144, 86936420624851435520, 40865461467829231616, 9905956521090535424, 30646625005150392320, 24305081955481804800, -64069135315262996480, -35202275251771125760, 40221623728344145920, 8726879722765299712],
 [-15242185296288919552, 47004677686683303936, -13795432918449559552, -5921964376682705920, -20809798109140307968, 63163346226952724480, 19376728383389769728, 62061312711175569408, 72381343475982360576, 23401210460013346816, -68370521702188990464],
 [18912062375920226304, -104390950741892546560, -59111927907518447616, 29726269447826755584, 49791661124021542912, -38745098709074116608, -42169468088561975296, -4060742774212236288, -9592155040128503808, -4680145465652499456, -16000143889830821888],
 [-56790383729952735232, 11783981489201432576, 5219530874176861184, 36377415090103865344, 61235877199334965248, 56720963719127359488, 14527108910915078144, -34521632022274375680, -82309091161390055424, -42605028825619341312, -10180688895519891456],
 [-11212489651353778176, 19637725109926277120, 15097484153674932224, -24341499112242827264, -79351551100392538112, -40516725983771074560, 65988582907023106048, 22042442389872238592, 41814140083047522304, -36216931852796043264, 67878893285605679104],
 [-17736286563247980544, 17235473735858335744, 56347287895727276032, 10415366950089117696, -95802327341768327168, 22994462257108586496, -25231026699174973440, 58942275756404842496, 59379789719674765312, 17557986827117545472, 15200895311542378496],
 [27332521775256367104, -43827878989238837248, -41421586968654839808, -39562035419771117568, -4154147474903825920, -34686069569978302464, 58841024853508317184, -20360164029812568064, -84821816299520458752, 55967659419467874304, 20122548950392561664],
 [-27448461453720166400, 46377129283916505088, -20882339576837918720, -96745815070395187200, 37471569899045322752, -16084257585857910784, 15706474081452361728, 14139061298831327232, -42797782388465377280, 70987919785195651072, 9880651711228463104],
 [-87033377575452524544, 69082402743138082816, -53942135710352539648, -31292120753740824576, -2667099116432847872, 57563750175589294080, 40214805539274391552, -7628204765454084096, 425556443401174336, 22598889400188334080, 5909630346689519616],
 [-44175857215224774656, 50158971690213515264, -25874420184881950720, 43814093628398051328, -85880190445906182144, 21578521338270097408, 51755629654061457408, 58703375161530269696, -18604053931999023104, 14574475113287757824, 14042256331909105664],
 [11300380289226758144, -49458114889401057280, 14314153335533875200, -2013156048902389760, -2322755098033367040, -56385980460581281792, -40507515708932792320, -68642150672214376448, -64042536140773769216, 24565564584933277696, -68942214328911446016],
 [77306610871347806208, -21118220199702945792, -60637821270508978176, -7748765894329867264, 14249658343195584512, 64413057687531905024, -6590791378092922880, 23253465230248062976, -52771160110417518592, 59473258070091276288, 18991332975165206528],
 [91289030830369570816, -23968621854946054144, -2602207141429089792, 14549785589967945728, 3680106999455950336, -8140131590691865600, -12436908752580485120, -51282290171707514880, 96739212602745094144, -11929474163914661888, -16919788219788394496],
 [32894961032652750848, -9424783468530972672, 8292367427488762880, -64432352897785520128, 2222195632225187328, -4431242969058149888, 37330000455458398208, 75997509507987259392, 11084333295707772928, -26128623644787843072, -91583749373964992512],
 [35904760438603530240, -3180889865322931200, -17434714381205542912, -86764206488051761152, 20298355427726041088, 33146881026931576832, -41805750549028986880, 69168568359778656256, 67815458762649993216, 2168054489846233856, 646000884874052096],
 [-28706203294023143424, -2186434840437546496, 61734373105643888640, -24824733993466847232, -52593115069774782464, 63509760527981608960, 41699630384749125632, 42412421252627013632, -33511344485676347392, 20734772563892523008, -68056634865121132544],
 [-52679471226927529984, 64440781367138058240, 13528962143015534592, -59345902230570860544, 10980932856346085376, -42250622267243208704, 50906378723403939840, -49497902473965838336, 56655860930411175936, 29698046295514791936, 10310350081453297664],
 [17807279438608992256, -65961140313969156096, -17659760082798854144, -4904831003460282368, -32089814453570748416, -87678628867084484608, 12946869736298215424, 43375168495079677952, -7265390807052735488, 33440895347679371264, -69546901744120102912],
 [-41164478596476428288, 29073870048784015360, -74165380599413653504, -32106459731474862080, 17988465858906658816, 65557853446510362624, -49668638763666669568, 12108753763042457600, 45135689993500221440, -55016622255501221888, -20195716052355235840],
 [26923978219894013952, -25073042170743902208, -8511811668877731840, 62659608172807749632, 43039908296297734144, -109102962134370353152, 36296049641176559616, -24458022872890691584, -9479377822212452352, -19070582516839919616, -17343647981943513088],
 [-16957305425767929856, 25500643892618158080, -14956428444245286912, -17827028138711914496, -34847110855341391872, 10111162387031752704, -128306601869112082432, 36701298485163507712, 32441432055445909504, 8334486315987973120, 7120705245698794496],
 [20960494491527905280, 61305529356430778368, -26021888666238496768, 6511065811164711936, 60267424650651164672, -1320304442916399616, -33282329748003303424, 90113657962527113216, 25983494545602342912, -56678411138654126080, -10542949089005158400],
 [18660653078104870912, -12438593814690457600, 61519081620864106496, 19216722281318842368, -46616656332274229248, -50826377897652699136, -52217185757709115392, 19479630773715664896, -13850350865077217280, 12857302357505488896, 94348220918627450880],
 [-33187084314136944640, 21628563679892598784, -97548736020344487936, 45205358348235194368, 42858672109058129920, 46857067448874377216, -14101350651815792640, -65857429738295754752, -6395384423755443200, 6340401923709156352, -111816680843217584],
 [61749605768625086464, 38574681587103326208, -10649271403804565504, -80658491981630242816, -47532582544299483136, 8136775656685863936, -54826305888231391232, 18881026550177943552, -21074869828488105984, 60227892342783074304, 9916281759436670976],
 [-47494682433886412800, 64374534233741778944, 47941492577685913600, 48633948925472555008, -910774047918568576, 70011739416149450752, -28197097920011902976, 66090873705088671744, -16668513250971496448, 18488406173630128128, 5420764381388526592],
 [-35482412342180220928, 34520399301840367616, 22480595330477936640, 1887933593299703296, -18940198927261032448, -13119964366328633344, 82483286395234140160, -19239577193503191040, -44091652952135262208, 88394628306422415360, -36827541741101756416],
 [1359677265111917824, -87327634575255158784, -2807528711828698624, 40206115940854923264, -45870406264287322112, -15753146377342326784, -67262526182740836352, 54782312151442636800, 49860950111501197312, 12683727145913094144, 1047281497959310080],
 [49872395387069808640, 45436543158961283072, 89412683722878451712, -9392621547612016640, -9838348167213201408, -59763277013428789248, -31535605890773790720, -23105365289713000448, -56419479405870006272, -27629590112567803904, -374476521593175616],
 [93319159416151769088, 54093796685028450304, 12772999297992677376, -1124132209273937792, 70754657365866332160, -40678409952551919616, 19354638133600579584, -40074117536532078592, 7657710015061209088, -34865507166949773312, -7957282399440762880],
 [-40162932481739161600, 4032232965191729664, -35069593490577223680, 1229185627558128128, -96851716493920862208, -73038233770439352320, -37532017171679502336, 50059992236159901696, 4198816356557176832, 4885773328087648256, 15696164952849659904],
 [-17695560682862288896, -37729245498619977728, 9780523596433205248, -43642640142231191552, -16104369695532902400, 40141776516514004992, -31087991718941986816, -6933972899888482304, -75389088266389667840, 90724429960480309248, -35233662781241659392],
 [-39226810086410289152, 55194060529069088768, -15702544520936669184, -7032519447195139072, -43611038986539425792, 35123969829004390400, 34883848822999990272, -50279515777038483456, 80951793293750386688, -54653225638977519616, -21822506679076016128],
 [-19908882249128001536, -72596572036435025920, 49314575865572663296, 38353863894214664192, 44507132857394110464, 2568311541218678784, -55122882850329321472, -21568146840350400512, -35354769831681249280, -73546769980021817344, -7629807016558358528],
 [-1266337534250004224, -68046521206485721088, -35340042843316043776, -44752659265411489792, -26166483697141547008, 22114745220392783872, 12681710141298245632, 86026633343023251456, -71647075839685885952, 5073542502726545408, 470247602939911104],
 [-30549610636557742080, 56502678414153957376, 28956864960195436544, -18305131017619968000, -45244841190453665792, -25651585050170679296, 11466458034013591552, -93545498571492851712, -16925633250900592640, 609056690285327872, -67856280558691131392],
 [57987057061081923584, 102011744689287757824, 31428665508323033088, 64733112455239557120, 11646670734687905792, 28750892773362847744, 12997883323836340224, 23225176263315210240, 32142954933340770304, -11127480875869534208, -1853871000457778176],
 [31754008530576240640, 34104083350801444864, 27957111936920256512, -9127203448438921216, 84095118121492611072, 11067432708429619200, -26046949871609831424, 13054498371115188224, 66905302459927453696, 35689839958018396160, -70344405007766593536],
 [-47024237039845122048, 60985394714498670592, 67623590104787779584, -55845435383454400512, 23639364214135955456, -34639585324392542208, 28121104598133190656, 22417268135974486016, -21379541311094710272, 4036452316057149440, -68047652957930930176],
 [-27521669726044147712, -9473645728154802176, 110623099836991307776, -22621735112589008896, -17374188121939210240, -3085137572636386304, 77711557975310745600, -34763638032336818176, 23111715443906232320, 9409102278366513152, -79010265394112656],
 [24803885254430371840, -8973986535076902912, 4360168121687695872, 64180868650255032320, -45782777095731453952, -76918741056769949696, -21805060348191469568, 31452257750061506560, -84531620968785854464, -17917841533479766016, -454742336824375744],
 [86814269394944950272, 28699763477444739072, 10553065092072894464, 34012050016370659328, -17048599265318582272, -11888937733366079488, -12080791694795694080, 31652415045831884800, -23929233126303068160, -99519923134031659008, -9635620238813726720],
 [89190456591500738560, -7846459453449651200, 10264019312968327168, 16055380684214214656, -85097183070306533376, 24096345437766352896, -70484197520261201920, 4681860730448956416, 21683697393400074240, -10777061379689418752, 86798910988354848],
 [-47084524885569536000, -45182305140389625856, -15544939436560592896, -32562399977480269824, 52376265435029594112, -66894877271286005760, 647674625247872128, -13276788028524431360, 10500084411144767488, -84014314217135177728, 40671564172144689152],
 [96982424076917325824, 51312922431827312640, -53874214493953925120, -7097618015999596544, -1526021491149772800, 6760039613353018368, -7944692367542791168, 30095266874679173120, 18812678340186816512, 19175142229617573888, -71050578306391736320],
 [-29399823377363648512, 4557604301500240896, -50239886021227921408, -11399522531536076800, 44040752160255344640, 51185576440899805184, 76677250501011668992, -21842332117052129280, 10134194161647245312, -75301286538443341824, 39369149104246767616],
 [-3480411372812345344, 33827218684607930368, -70637261300105445376, -31988471113824292864, 27814891953478189056, 43101274530808832000, -40814475674138427392, 2355830413580443136, 27652762101786435584, 90835669145131548672, -35919897613195165696],
 [-73391671154193670144, 54193204119792508928, 5058815010468375552, -13215050766161471488, -53285101764783054848, 11838147555387666432, -32555665408031952896, 18739472766288109568, -3821320636509436928, -25673563369396744, 94071862910327390208],
 [-67722503769890758656, 50864639605478268928, -39046422164745945088, -45662066483117326336, 16006365225842253824, -32733794645262422016, 41765854594730074112, 45658678501258608640, 42565614349733478400, -59753347380593229824, -21348873136983904256],
 [-77057923515882733568, 2508531106200562688, 27730063495856107520, -94510494244367007744, -61606039312401530880, 9996053025477459968, 29060038945720950784, -19061030980968628224, -22627941137781919744, 15966858645393520640, 16207005203936618496],
 [26084943977763868672, -77036173288826748928, -31501855247808241664, 65543311571333603328, 11377021333229574144, -24621819158485884928, 53669881099267235840, 24809488780590137344, 74641789909244936192, 4657010796456197120, -7075351268470172672],
 [3529929347912311808, -50800545842121670656, -1172752692897826560, -66791274015772082176, -75975372050413846528, 12110478269732505600, -38921685703085645824, 34510194680622022656, 15354759148430172160, -32392070367253180416, 69373475782784573440],
 [43926224735497969664, -30206766573651431424, -6981221743536806912, -93618655992538400, 6943339865212061696, -5232184827031739392, -90953559310634139648, 20227561500209434624, 32832424353927852032, -28593793128964239360, -90744147351672668160],
 [-39227732506312474624, 30524399678345043968, -29479909343580864512, 15765512807030648832, 103507921962402742272, 75026821815174070272, 5068972944953679872, 38936965540284375040, 10391479877400307712, 14036028304284624896, -460915601510407616],
 [-18045325513036855296, 20260799712868982784, 10943885854699776000, -76019577706787520512, 12850068249364121600, 28660627143003631616, 112899325817974554624, 34180464547658788864, 12952085753122729984, 8603199370362834944, -550979937594170688],
 [92669151823215968256, -25059987327309750272, 26942942498124324864, -27550174445737312256, -35740416840735686656, -3850721840477273600, 22064739929862373376, 12377857464051529728, -64804624402318721024, -25207036906982866944, 65652562358543056896],
 [-58869377724080717824, -3888606266001132032, 46291679793250459648, 56767842483332030464, 30770719931542978560, -66033412485210923008, 40213637142025814016, -34670639914397106176, 36387095570048036864, 58201533393130815488, 9693356755379556352],
 [-52590424833372274688, -63884895785483165696, 25570878671284559872, 50951122625766449152, -53318722632699961344, -20239982799483867136, 87442519267555442688, 19794269364276617216, -5001299806118728704, 19119760286413467648, -153169275197853472],
 [-12106502739686295552, -15022373207232292864, -25465400205796290560, 62076889733994987520, -10121376951212988416, 34157319583894634496, -75213827513840467968, 43361863579473035264, -28483519435310268416, 84792379536647847936, 9808731267021723648],
 [39081091231614795776, -42293172792802426880, -53231199345115381760, -2483180359210977792, -27403720613696761856, 33762507666823753728, 92525631152551084032, 29845361889760464896, -272035895696483776, 113957048426519328, 65418687652051132416],
 [2629069149392472064, 41207629263382913024, -6354661993959872512, -82771616953292423168, 34906110494533156864, -78884074643622051840, -22027634731994255360, -63413350898128199680, -27073451319549894656, -22376756995172544512, 523355036177287616],
 [-59349323773587537920, -75172800647825588224, 10266969835122399232, 62790057033196371968, -20385889242535014400, 28018798630239076352, -3651679020173891584, -45838515502339178496, 66577961266384306176, 28429996330260795392, 863668773707091456],
 [3426372747240830464, 25550529523093229568, 702816243435297408, -6147881371766063104, -11081138750582784000, -10904711181944696832, -11613204219657910272, 985286452754179584, -12104511229363056640, 116718018414926118912, 83234937492447576064],
 [-6326757202987667456, 1632195058538455040, -640525101510024576, 19731709613276024832, -62609521875858669568, -21688497768573493248, -48909355668529586176, -39183220485966659584, 84842464245031092224, -33354198802413662208, 68681071682037104640],
 [2631951631853216768, 21304884621713862656, 23058308206968123392, -49346195767032373248, -46685634014105305088, 45728157944279949312, -75075241225331179520, -52990612054597550080, 24456155581687427072, -67770193173428674560, -21087942026646990848],
 [55057607690613547008, 74874985622934454272, 8904759993335789568, -52472662827038457856, 43556187654266404864, 73849767779639738368, 19929942928821825536, 27922198238369632256, 3167161361018689536, 41141607097621962752, 8794802001285447680],
 [2116009323353306624, -16110596693643769856, -12884129353515118592, -1486349035521027840, 88831067669942337536, -32173854376457359360, 58032990512793903104, -55936771698520580096, 75071777230563672064, -1151474518050002816, -16907664065562198016],
 [4122154761982024192, -916305221370841088, 40125607041455022080, 59742538586249527296, 52622954635833458688, 61147574968119689216, 7714863222084525056, 47915964908594806784, -2996517046591645696, -78665151602110939136, 38989218675003875328],
 [24806280297629208576, 2197010518382354432, 95165091733302083584, -42698982374741630976, -78007364981702115328, 37512433968986906624, -24853656515355803648, -31012400313634119680, -22665984483734466560, 24084531852436729856, 10615030326896805888],
 [61975792691229106176, 55892562521764085760, 22267053205564801024, 22167463078881275904, 681980964164605312, -65886983293733896192, -76292389887572656128, 14594067651366516736, 54929915542353338368, -20818764926908448768, -290094200689260224],
 [22560355501996568576, 33116090839308062720, 61987549506296233984, -24251277043542274048, 82663500608723779584, -28633221817741463552, 65329922908841697280, -1593235271995663616, 3611792849795893760, 58607728217120284672, 19242708573037707264],
 [6969016543859016704, 55196614984128413696, -29511969332701786112, -97993782332104671232, -18653321086853267456, 27443493314085437440, 17175308717894223872, -10574418539332843520, 10235184231274291200, -44109800465332436992, 68196521266191507456],
 [49371126889922174976, -11867293181549373440, 82881194300461252608, 71879147006096277504, 44039511877984575488, -13052481953855623168, 48583798863674630144, -15661700552138518528, 45712955218134056960, -6395680583278510080, -17824905312271312896],
 [76924315984093888512, -2413179545400150528, -27716446677911527424, 94483775324219588608, 61389443874821521408, -9992737581453670400, -29136860620478644224, 19157594178891046912, 22600566451255205888, -15842426048589436928, -17699030754233901056],
 [-38920227733911920640, -47751601006321893376, -53699643414274351104, 31503871058573414400, 5282587983009251328, -36047552032644616192, 50320619541705957376, -56088255707816198144, 1152139070194369536, 47295665555353952256, -69737579793151557632],
 [-87830865703843119104, -79480741173456830464, -13070891778654660608, 24150948286301503488, 39225844639952281600, -39850958563579027456, 3222060457271819264, 50100276216914280448, 28177118776957202432, 23461846715394965504, 955180366193437184],
 [-6041671358560622592, 8895881015825620992, 68069669290179305472, -29440309475182661632, 60649567925528584192, 28169355873266241536, 19170666562695274496, 104793323179633508352, 16787262907589591040, 7697119333486970880, -354556764038471872],
 [25964262461011800064, -16995215061302941696, 273556297481211968, 87964180770514747392, -12891258049477885952, 22089722775115726848, 68192428207893872640, -42971442230767894528, 13904679636866535424, 73213306866212986880, 19310576749266018304],
 [-9276492932555718656, -32596581437841883136, -38367793816068644864, 61166220172372148224, 79656115093771419648, 2480608464806391808, -16024189816775727104, -12665554179254534144, 47086994965225783296, 77523194825232039936, 20508903066361016320],
 [-1641072477155787264, -90610199688604155904, -15997242240964067328, -30320390529665359872, 31868408255911378944, -10289389766526869504, -21408030267569676288, -14963078897234550784, 77326005055515082752, 7346415983153406976, 67475824360066080768],
 [-99034638046223368192, -49840588389442322432, 54036256805006401536, 7025537138260575232, 641814142362147072, -6164151128456402944, 7194241457476112384, -29895643108025643008, -19648856215408107520, -18886479550392836096, 69210985728497000448],
 [13795783213072111616, 42315311153475182592, 15441387559459246080, 34671818348845150208, -73557675580062187520, 91382090931704594432, 125579084006561984, 18783238792419397632, 3450149859790486016, -61392457249522565120, -21480444652403310592],
 [-14425051823493877760, 21550343443113377792, -25119753537884393472, 79945494622279204864, 57725761685609611264, -1893333677229312768, -16735011868543045632, 70389160350924070912, -74140672436237336576, -3880965625044567040, -918660374017730944],
 [-33124251626414059520, -33288605784315015168, -28113765760199671808, 8181995442314000384, -85059172795142750208, -11149140407176194048, 25964044984406532096, -12792398273374361600, -67774280231353606144, -35113826670629167104, 68497577396184449024],
 [-60709001705087836160, 12154432901044574208, 13074294581751384064, 22583795637446303744, 25484641801656344576, 43772357013685338112, 63610664839827677184, -100620965661713317888, 16716632341256255488, 15745881143419142144, -182108190660343232],
 [-63033609064758861824, 28365716084077006848, 25986112589990281216, 24078682574684327936, -63465094179681263616, 75976431344878239744, 5437991754011012096, -44613531396969914368, -58350958699331313664, 17080573340713134080, 15113078979288850432],
 [35671652494471368704, -35465454043662585856, -62255173467231010816, 60974776542711070720, -22158822311981580288, 43381459879374462976, -27639920431447056384, -23979164279907774464, 33351217968769482752, 44130646413014228992, -69874377145645367296],
 [-19248642274568179712, 67720760763631673344, 17400767282482933760, 4761826623590146048, 30635261495160152064, 87954387735819665408, -14354597097767387136, -43189380829632479232, 7444149519392555008, -33298691504050241536, 67719341839783788544],
 [-13732602274115512320, -42384937694151335936, -15447739787664627712, -34654279201929601024, 73616722551285702656, -91381638744837062656, -83929771060120640, -18799133801086656512, -3435286978183439360, 61281937286506913792, 21511280618756538368],
 [22143340859487707136, 11034106650209019904, 38129692609073987584, 58498985400588574720, 33359599731098869760, 66145035871447818240, 5573468617406158848, 42794369187786473472, -7561677750467294208, -18667929042052304896, -92378930937777307648],
 [83977419932272705536, -20720462998955048960, 21350553314637733888, -15895033658305126400, -45154321134795227136, 3043386255870563328, -2776081278449142784, 24475472349885829120, -74803392349107388416, 19926542889156571136, -70288574506947952640],
 [-33101200450805346304, -35369154232748343296, 35230097335313580032, 15578976750094682112, 47136234885772386304, 27813403507517947904, 69104401006958067712, 19922816574490701824, 38150996350598676480, 58286805832543289344, -70192189710152835072],
 [62762121558552870912, 25069436363331031040, -71981198403994730496, 49421173620078682112, -75170426567336034304, 22696548938976919552, 6508961211959748608, 4458450127266726912, -56290748770001928192, -13750620520537849856, -1032925812975162880],
 [-5648070801252884480, 10749817044626231296, -61508408060278792192, 8779595133999544320, -15692196560369588224, 3658030991733515264, 18496018526186172416, -76372785044665565184, 89179202763763597312, 57307542326741803008, 9931305367735265280],
 [-15267949430955511808, -12301805839788462080, 48467095814632407040, -43597565020205080576, 68149953597524434944, 79713351027473711104, -64623179314077753344, 11649141876562944000, 14817681396061030400, -39174030102629081088, -8897139364293063680],
 [43922303520541368320, 58248696555279360000, -55873625737123446784, 51856320516176715776, -19844297890054082560, 25051228106984071168, -50632776766076575744, -43590132531420258304, 40348583634400485376, -58506965432255422464, -10432685395001214976],
 [-47671433683330351104, -34163886334902820864, -6390463118203494400, -26669071397151072256, 64844957352337604608, -68058523455650258944, -7057997926950584320, -2832909389229752320, 1709728379574921728, -21666351905520779264, -90659977832560099328],
 [79054284885997748224, -32525193830820507648, -21417677168533331968, -2728675543984249344, 29181467219135430656, -9232799907003121664, -90667311963419721728, 13395669840246269952, 32410111809552207872, 56210412209480671232, 21391748041211576320],
 [-86351808402619416576, 21960523987431374848, -21350222705749196800, 15871304538266464256, 43656825920253296640, -2403683383777881088, 1239083123962383616, -23514239670562947072, 74695997150460215296, -19879304104179884032, 68488043320873164800],
 [46864794417296490496, 33543329114403074048, 5588250074448018432, 25415205817239576576, -64984771760606208000, 66564005198380351488, 7099563767410844672, 1984556165924566528, -1293926782855546880, 20099944162896261120, 93089391199523651584],
 [100743974056065875968, -16013298544737665024, 22038248833308651520, -23847752379019890688, 71195414891893547008, 29275195349312131072, -26188251665140051968, 50574530199186644992, -20895441168452034560, -20502445404298141696, -17073260238839953408],
 [-15446315290918260736, 35982075748254224384, 46567401978484555776, -20860174207373750272, -48250740329395142656, -9910023245758697472, 13213716697334941696, -90610530534225756160, -14772218907521906688, -44126515567477612544, 68297308683700592640],
 [-27219747278120067072, 43832930755818250240, 41416139693876617216, 39538604991115870208, 4194983660224755200, 34623455173358297088, -58851483538248843264, 20332298998478839808, 84784053812125433856, -55507766794368786432, -21783115526525464576],
 [-79452130881446723584, 34590977584461828096, -64541883874331541504, 16970362867838144512, 6630653979750538240, 1967921055242335744, -32470233292401418240, 89001285320818999296, 14571447822994489344, 19000582648851779584, 14782229338676746240],
 [-61960410909577707520, -37738037433722109952, -6217596800644531200, 30260191036273889280, -28672890068621770752, -65003161332762902528, -9849494652144154624, -282690929639159232, -60550734181781053440, 81484977807280881664, 10471424763116380160],
 [12103340579786260480, 10453625751954358272, -67145424901433344000, 31560725051677638656, 10520630616180979712, -6122783149377893376, -44756724466480676864, 37699825443555082240, 111349918815672680448, 6803270742404293632, 112289159964721040],
 [-49977467143754285056, -42595319616490610688, 48631146102955302912, 1618739884034509056, -100002114532114448384, 25496505782900051968, 12376571892476585984, -33195331919149993984, 19471853352717123584, -48820418977248313344, -20740386311188328448],
 [-75581624303891791872, -31189675370578866176, -22074727669853630464, -70288965513836453888, 30974643966557892608, 31081974991923351552, 11506201770362003456, 49756751074137145344, 5464782895183950848, -18251096560131524608, 69052518806149242880],
 [59349352039195779072, 75172797571169648640, -10266971324563816448, -62790058995471982592, 20385900221601562624, -28018812149448065024, 3651679430061918208, 45838508326496944128, -66577968728898756608, -28429904297255223296, -863969426902861312],
 [-9291349571300032512, 43626860728771969024, 51956630744122482688, 99883859519920898048, 26593290170576236544, -44555229357005291520, -28808863784132435968, -1882131766245384704, -12481339375363119104, -57597383471912787968, -10632753409927882752],
 [2656095453843342336, -112351113624980488192, -19213440054885593088, 16452638979593840640, -6839516689881368576, 21197538890565115904, 18473371633361838080, 16867723444266086400, -7548946329017576448, 82884174193125605376, 21441202842933551104],
 [-41846961805306036224, 33896201290450219008, -38767216364326494208, 42304936817031569408, 3072806672578932736, -10608728685899122688, 109949011638953541632, 2711167500095422976, 56486631679055781888, 13239816666282504192, -1091286568508978560],
 [3884957577529091072, 89967949320285732864, 15381673070159650816, 30252294229265186816, -30874919767195967488, 9846696835359397888, 22674142934474014720, 14664467270730598400, -76887742371434364928, -5279261420828490752, -69267398641936834560],
 [-29843534249145774080, 22530728712549335040, 18357313858313373696, 88884279812397039616, 50872025591980466176, -11100322157247037440, 16220423152474185728, 73325274699813675008, 60320792451370450944, 12333643010765901824, -894489534848738944],
 [48812177824934043648, -48435363953127022592, 49844584565853749248, -19917796869592576000, -24544290275258982400, 56939492647128154112, -39894339261507108864, 75801981559064199168, -4541568298959700992, 53501119641705955328, 10160447235362709504],
 [24187437523694170112, -44944186397636747264, -9119742620194248704, -34126670879810580480, 28040416970518990848, 112625370673363877888, 38809444121017606144, 51288414043968323584, 8263194063436443648, -11904793467200552960, -10384706653855455232],
 [29331347822248042496, -79829826310263570432, -24894724849298251776, -7665495557429917696, 3036325983334716928, 57785275428686970880, 63818271967194177536, -65895906032630005760, 41790233405281525760, 8901350750820539392, -16654667449005961216],
 [-20495678457922895872, 55067144066182012928, 56720178770495356928, 27731105851846864896, -28727490277100883968, -13591083495644170240, 32276931342776512512, 85971628667209367552, 13239828110194083840, -65215143114990600192, -21637928728485957632],
 [19784297547706216448, 50588174574508867584, 74900235085598228480, -23349995237561946112, -7163251668322470912, 3023877672061121024, 6071248596189820928, 54520279337322618880, -72107326366841675776, 65726314949527216128, 9112092593715827712],
 [27521866709414428672, 9473527405040990208, -110623112490460938240, 22621781370400555008, 17374324633440968704, 3085109267828897280, -77711461152168083456, 34763596273204301824, -23111709764328849408, -9409096025951985664, 80701039519898560],
 [-59014474335322054656, -17748446524363681792, 16781900780661542912, -18502055538542477312, -74433969093426020352, -18645253039565897728, -83853091402032365568, -53263946426962550784, -35554614892946526208, 7393083483131473920, 16767803336676052992],
 [41846910378953302016, -33896249983717793792, 38767216881457324032, -42304910671970476032, -3072796587947514368, 10608780296227213312, -109948975675416952832, -2711154732313142272, -56486590045643726848, -13240267721238931456, 1092604871250546432],
 [-2386304679979585536, 36504761341995642880, 330140524167411136, 17916687009620086784, 67945207102973976576, -36129108179419860992, 58453955931029913600, 26904355376946941952, -45646859450268753920, -88752203943079985152, -9772428323768221696],
 [-61975824250835132416, -55892444923159339008, -22267046272698900480, -22167505611194236928, -682055041816346112, 65886942492583903232, 76292314871100211200, -14594059145009909760, -54929965927170629632, 20819239034611273728, 288973333957161088],
 [32176558238869262336, -57189044058247340032, -29848009823851835392, 18801667916137844736, 46959008721300570112, 24779203814920183808, -10518877183185465344, 93027252526407991296, 17022397850356150272, 1359466957468658432, 65971685631725846528],
 [1527247732436026880, -98353189637886918656, 35693351005452374016, -52618032267697250304, 40346471665195827200, -16828307619026395136, -39520880671309938688, 63553961893059395584, 4599839717762280448, -21155958805240496128, -8885243201246640128],
 [-32383833535337656320, -21523242250403438592, -53703384231170752512, 70974330322868166656, -55784623015300235264, 55514402640392544256, 58430471276444704768, -40157237287382548480, 392834549972659584, -31909660051413012480, -10286998574119200768],
 [-75843602220980764672, 27374701301272449024, 67209774650937712640, -17691946098433468416, -29430696259337420800, 37263081401306800128, -34269366837032398848, -53593503249168080896, 58875855583184453632, 25103460956443299840, 6955581497543049216],
 [35546908738411188224, 88807231887601844224, -59978400746546577408, 55534686992796909568, 10608352148421216256, -67287404795949015040, 701259360274038144, 14369355041404932096, -23043621245427535872, -23794702581563342848, -1398517615428215808],
 [-9115945612333735936, -9472860823222654976, 18697668123046027264, -74552195811831037952, -39018452828155813888, -71681903145410183168, -28458242982166630400, 26958189822749831168, -85010265980061188096, 12333117506595328000, 11018240854013308928],
 [-12948348104914020352, -39360479749123039232, -59601670217903357952, 11637242936162074624, -25774493668987527168, -51353173069445955584, 84965111273612886016, 11101428485966274560, 470020988275788800, -75173061212839329792, -8853534553818013696],
 [-106929637203434504192, 25081203479228948480, 46089763366725132288, -5648504917085511680, -10723817826159413248, -1129339035191763328, 45257772383327035392, 54252848678875848704, 37689303063498522624, 28343927572583661568, 15070185319536828416],
 [-28500738318056722432, -24231988866281439232, 6074002142278431744, -5197210072577627136, -21973054160521662464, 56752367575123533824, -46159358380176687104, 106360878298408337408, 13148644040235608064, -50567952301529456640, -20921564143702908928],
 [-11471736176404785152, 85692438808020271104, 2576560572884022272, 11247290572301709312, 12321044981087309824, 77236546588998729728, -33097345625354072064, -45753887984701014016, 10455502620520507392, 9157272670261850112, -68343386396802949120],
 [50558158899588685824, 34409951209603473408, 38320631292783493120, 29472524007576510464, 36627968497549672448, -63967481757159170048, 259521266230650176, 65815584150572449792, -24114305174155681792, 18054885056333922304, -70909202505130795008],
 [-8636602899598294016, -113849198493594681344, 51480597341652639744, 7125833504934900736, 32437481774079123456, -41804406098562105344, 35556817092770562048, -38883152076451127296, 13525468113287753728, 4768650355363770368, -16098059066412673024],
 [-28649325029777321984, 13799134418565040128, -31735830238849789952, -36560999074302648320, 64417200045045678080, -22818669430261858304, 35842907951922769920, 57336704860359254016, 74759436252779053056, 59132047187820036096, 9744257229542897664],
 [86675016369126785024, -29160785742826496000, -12801046950273144832, 65384993469454925824, -38369160774763937792, -21678330479264153600, 4589150843173566464, 57649041376718602240, -2806902231277437952, 57416105305857802240, 19621283967051624448],
 [-5204489742219787264, 51642688429240115200, 1965741846235028224, 65793653495875510272, 74628457865129197568, -12263537948558241792, 38099408157339238400, -34446556175904423936, -15561903983755997184, 32679102522563493888, -71297410205280493568],
 [-338813839236509248, 19545379858329571328, -64680926588033228800, -26175329803739037696, 26096941847784124416, 43322843815260012544, -54634346150435831808, 8876700742360065024, 37032824190464442368, 25716628276139307008, 94195344135880114176],
 [-1359603005813828352, 87327661215294767104, 2807525983760724992, -40206136739266404352, 45870415837875429376, 15753093131321726976, 67262502423074111488, -54782331197720133632, -49860987513559146496, -12683299054243928064, -1048679591736599424],
 [20209237090519834624, -30708662754124226560, 61048762028654632960, -2761960658353753600, 34563189398315212800, 17890620394305060864, 3528953210000444928, -76854435745913028608, 58955741588415332352, 73038776136931532800, 21162892742352265216],
 [-56720226053784887296, -33965135626119868416, 3912019699448901632, -19981494975717265408, 14519906563838412800, -50865265663573663744, -25679469849631576064, -109252050866207047680, 39504165520108265472, 6053668714671792128, 1385003764377766144],
 [37466250678995107840, -8423374817422732288, -64621229326878515200, 77347434471675707392, 22236242957051699200, -32265991685222993920, -11491115252859500544, -32858140168245751808, -64790737390451408896, 52856836889060442112, 20147871295874011136]])

In [ ]:
#@title Verification

def is_valid_kissing_configuration_exact_integers(centers: np.ndarray) -> int:
  """Returns whether `centers` is a valid kissing configuration.

  Using exact integer arithmetic, verifies that all pairs x, y satisfy
  2 * <x,y> <= ||x|| ||y||, hence the angle between any two points is at least
  60 degrees, thus `centers` is a valid kissing configuration after normalizing.

  Args:
    centers: A set of points. Shape (n, d). Only contains integers.

  Returns:
    True if the angle between any two points in `centers` is at least 60 degrees.
  """
  (n, d) = centers.shape

  # Convert to Python integers to allow arbitrarily large values.
  centers = centers.astype(object)

  # Compute the square norm of each center.
  norm_squared = np.zeros(n, dtype=object)
  for i, j in itertools.product(range(n), range(d)):
    norm_squared[i] += (centers[i, j]) * (centers[i, j])

  assert (norm_squared > 0).all(), 'One of the centers is zero ... this is not allowed!'

  # Check all pairs x, y satisfy 2 * <x,y> <= ||x|| ||y||.
  for i, j in itertools.combinations(range(n), 2):
    dot_product = sum(x_k * y_k for x_k, y_k in zip(centers[i], centers[j]))
    if (dot_product >= 0 and 4 * dot_product * dot_product > norm_squared[i] * norm_squared[j]):
      return False

  return True

num_spheres = sphere_centers.shape[0]
dimension = sphere_centers.shape[1]

if is_valid_kissing_configuration_exact_integers(sphere_centers):
  print(f'Verified (using exact computation with integers) the sphere packing is valid, hence the kissing number in dimension {dimension} is at least {num_spheres}.')
else:
  print(f'Could not verify that the sphere packing is valid.')

# Novel kissing configuration with 4320 points in dimension 16

This section contains a new 16-dimensional kissing configuration, discovered by AlphaEvolve, that did not exist in [Cohn's database](https://dspace.mit.edu/handle/1721.1/153312).

To show that two sphere configurations are distinct, we show their *signatures* are different. The notion of signature is defined as follows: for the set of $n$ sphere centers, we normalize the centers so they all have norm 1, then create an $n \times n$ distance matrix, where we store the pairwise distances between the centers. Then we compute the eigenvalues of this matrix, sort them, and put them in a vector of length $n$.

This gives a one-sided test: if two sphere configurations have different signatures, then one cannot be obtained from the other by rotation and translation.

In [ ]:
#@title Data

alphaevolve_solution = np.array([[-0.7071067811865476, 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865476, 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865476, 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865476, 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865476, -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865476, 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865476, -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865476, 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. ], [ 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. ], [ 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. ], [ 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. ], [-0.7071067811865476, 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865476, 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865476, 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865476, 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865476, 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. ], [-0.7071067811865476, 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865476, 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865476, 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865476, 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , -0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , -0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. ], [ 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. ], [ 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. ], [ 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. ], [ 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865476, -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865476, 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865476, -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865476, 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , -0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , -0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0.7071067811865476], [ 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, -0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, -0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476], [ 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476], [ 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476], [ 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476], [ 0. , 0. , 0. , -0.7071067811865476, -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865476, 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865476, -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865476, 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. ], [ 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , -0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , -0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0.7071067811865476, 0. ], [-0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , -0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , -0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865476, 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865476, 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865476, 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865476, 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , -0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , -0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , -0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , -0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476], [ 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476], [ 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476], [ 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476], [ 0. , 0. , -0.7071067811865476, 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865476, 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865476, 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865476, 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, -0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, -0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. ], [ 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. ], [-0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476], [ 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , -0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , -0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0.7071067811865476], [ 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , -0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , -0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, -0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, -0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0.7071067811865476, 0. ], [ 0. , -0.7071067811865476, 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865476, 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865476, 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865476, 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. ], [ 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. ], [ 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. ], [ 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. ], [ 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. ], [ 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. ], [-0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. ], [-0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. ], [ 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. ], [ 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. ], [ 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476], [ 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476], [ 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476], [ 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476], [ 0. , -0.7071067811865476, 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865476, 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865476, 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865476, 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865476, 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865476, 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865476, 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865476, 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. ], [ 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , -0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , -0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0.7071067811865476, 0. ], [ 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. ], [-0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. ], [ 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. ], [ 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. ], [-0.7071067811865476, -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865476, 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865476, -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865476, 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865476, 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865476, 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865476, 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865476, 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , -0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , -0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , -0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , -0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0.7071067811865476, 0. , 0. , 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, -0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, -0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. ], [-0.7071067811865476, 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865476, 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865476, 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865476, 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , -0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , -0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, -0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, -0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0.7071067811865476], [-0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476], [-0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476], [ 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476], [ 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476], [ 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476], [ 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476], [ 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476], [ 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , -0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , -0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0.7071067811865476, 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , -0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , -0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , -0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , -0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0.7071067811865476], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. ], [-0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. ], [ 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. ], [ 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. ], [-0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. ], [-0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865476, -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865476, 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865476, -0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865476, 0.7071067811865476, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, -0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865476, 0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, -0.7071067811865476, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865476, 0.7071067811865476, 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. ], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [-0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, 0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , -0.3535533905932738, -0.3535533905932738], [ 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0. , 0. , 0.3535533905932738, 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , -0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0. , 0.3535533905932738], [ 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0. , -0.3535533905932738], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [-0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. ], [ 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , -0.3535533905932738, -0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , -0.3535533905932738, 0.3535533905932738, 0. ], [ 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, 0.3535533905932738, 0. , 0. , 0.3535533905932738, -0.3535533905932738, 0. ]])
known_solutions = list()
known_solutions.append(np.array([[-0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ]]))
known_solutions.append(np.array([[-0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ]]))
known_solutions.append(np.array([[-0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ]]))
known_solutions.append(np.array([[-0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ]]))
known_solutions.append(np.array([[-0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0. , 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , -0.35355339059327373, 0. , 0. , -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0.35355339059327373, 0. , 0. , 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , -0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0. , 0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , -0.35355339059327373, -0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0. , 0.35355339059327373, 0.35355339059327373, 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , -0.35355339059327373, -0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0.35355339059327373, 0.35355339059327373], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, -0.35355339059327373, -0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0.35355339059327373, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.7071067811865475 ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0. , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.7071067811865475 , 0.7071067811865475 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ]]))
known_solutions.append(np.array([[-1. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , -0.5 , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , -0.5 , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , -0.5 , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , -0.5 , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , -0.5 , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. ], [-0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. ], [-0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. ], [-0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. ], [-0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 ], [-0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 ], [-0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. ], [-0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. ], [-0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. ], [-0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. ], [-0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 ], [-0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 ], [-0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. ], [-0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. ], [-0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. ], [-0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. ], [-0.5 , -0.5 , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , -0.5 , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , -0.5 , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , -0.5 , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , -0.5 , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , -0.5 , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , -0.5 , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , -0.5 , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. ], [-0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. ], [-0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. ], [-0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. ], [-0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 ], [-0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 ], [-0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 ], [-0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 ], [-0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. ], [-0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. ], [-0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. ], [-0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. ], [-0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , -0.5 , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , -0.5 , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , -0.5 , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0. , -0.5 , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0. , -0.5 , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0. , -0.5 , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. ], [-0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. ], [-0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 ], [-0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 ], [-0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 ], [-0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 ], [-0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. ], [-0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. ], [-0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0. , -0.5 , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0. , -0.5 , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0. , -0.5 , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. ], [-0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. ], [-0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 ], [-0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 ], [-0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 ], [-0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 ], [-0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. ], [-0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. ], [-0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 ], [-0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 ], [-0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. ], [-0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. ], [-0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. ], [-0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. ], [-0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 ], [-0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 ], [-0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 ], [-0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 ], [-0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 ], [-0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 ], [-0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 ], [-0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 ], [-0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 ], [-0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 ], [-0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. ], [-0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. ], [-0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 ], [-0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 ], [-0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 ], [-0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 ], [-0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. ], [-0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. ], [-0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0.5 , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0.5 , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0.5 , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 ], [-0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 ], [-0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 ], [-0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 ], [-0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. ], [-0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0.5 , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0.5 , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0.5 , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0.5 , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0.5 , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0.5 , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. ], [-0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. ], [-0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 ], [-0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 ], [-0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 ], [-0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 ], [-0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. ], [-0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. ], [-0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0.5 , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0.5 , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0. , 0.5 , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0.5 , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0.5 , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0.5 , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0.5 , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0.5 , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. ], [-0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. ], [-0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 ], [-0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 ], [-0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. ], [-0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. ], [-0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. ], [-0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. ], [-0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 ], [-0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 ], [-0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. ], [-0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. ], [-0.5 , 0.5 , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0.5 , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0.5 , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0.5 , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.5 , 0.5 , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [-0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [-0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [ 0. , -1. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , -0.5 , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , -0.5 , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , -0.5 , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , -0.5 , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. ], [ 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. ], [ 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 ], [ 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 ], [ 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 ], [ 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 ], [ 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. ], [ 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. ], [ 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , -0.5 , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , -0.5 , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , -0.5 , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , -0.5 , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , -0.5 , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , -0.5 , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , -0.5 , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , -0.5 , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. ], [ 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. ], [ 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 ], [ 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 ], [ 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 ], [ 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 ], [ 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. ], [ 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. ], [ 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , -0.5 , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , -0.5 , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , -0.5 , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , -0.5 , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. ], [ 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. ], [ 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. ], [ 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. ], [ 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 ], [ 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 ], [ 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 ], [ 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 ], [ 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 ], [ 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 ], [ 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 ], [ 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 ], [ 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. ], [ 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. ], [ 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. ], [ 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. ], [ 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , -0.5 , 0. , 0.5 , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0.5 , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0.5 , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0.5 , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. ], [ 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. ], [ 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 ], [ 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 ], [ 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 ], [ 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 ], [ 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. ], [ 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. ], [ 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0.5 , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0.5 , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0.5 , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0. , 0.5 , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0.5 , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0.5 , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0.5 , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0.5 , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. ], [ 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. ], [ 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 ], [ 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 ], [ 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 ], [ 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 ], [ 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. ], [ 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. ], [ 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0.5 , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0.5 , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0.5 , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , -0.5 , 0.5 , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -1. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , -0.5 , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , -0.5 , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , -0.5 , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , -0.5 , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. ], [ 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. ], [ 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. ], [ 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. ], [ 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , -0.5 , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , -0.5 , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , -0.5 , 0.5 , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0.5 , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0.5 , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0.5 , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. ], [ 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. ], [ 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. ], [ 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. ], [ 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0.5 , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , -0.5 , 0.5 , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -1. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , -1. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , -0.5 , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , -0.5 , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0.5 , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0.5 , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -1. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -1. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , -1. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -1. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , -0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , -0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , -0.5 , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , -0.5 , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , -0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0.5 , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0.5 , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , -0.5 , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , -0.5 , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , -0.5 , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0.5 , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0.5 , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0.5 , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , -0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , -0.5 , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , -0.5 , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0.5 , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0.5 , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , -0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , -0.5 , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , -0.5 , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0.5 , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0.5 , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , -0.5 , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , -0.5 , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , -0.5 , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0.5 , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0.5 , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0.5 , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , -0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , -0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , -0.5 , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , -0.5 , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , -0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0.5 , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0.5 , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -1. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , -0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , -0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , -0.5 , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , -0.5 , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0.5 , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0.5 , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , -0.5 , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , -0.5 , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , -0.5 , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , -0.5 , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0.5 , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0.5 , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0.5 , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0.5 , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , -0.5 , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , -0.5 , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , -0.5 , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , -0.5 , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0.5 , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0.5 , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0.5 , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0.5 , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , -0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , -0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , -0.5 , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , -0.5 , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0.5 , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0.5 , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -1. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , -0.5 , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , -0.5 , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , -0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , -0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0.5 , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0.5 , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , -0.5 , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , -0.5 , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , -0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , -0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0.5 , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0.5 , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -1. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -1. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , -0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , -0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , -0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , -0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -1. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -1. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -1. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 1. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 1. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 1. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , -0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , -0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , -0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , -0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 1. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 1. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , -0.5 , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , -0.5 , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , -0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , -0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0.5 , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0.5 , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , -0.5 , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , -0.5 , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , -0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , -0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0.5 , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0.5 , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 1. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , -0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , -0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , -0.5 , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , -0.5 , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0.5 , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0.5 , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , -0.5 , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , -0.5 , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , -0.5 , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , -0.5 , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0.5 , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0.5 , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0.5 , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0.5 , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , -0.5 , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , -0.5 , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , -0.5 , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , -0.5 , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0.5 , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0.5 , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0.5 , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0.5 , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , -0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , -0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , -0.5 , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , -0.5 , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0.5 , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0.5 , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 1. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , -0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , -0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , -0.5 , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , -0.5 , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , -0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0.5 , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0.5 , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , -0.5 , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , -0.5 , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , -0.5 , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0.5 , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0.5 , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0.5 , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , -0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , -0.5 , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , -0.5 , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0.5 , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0.5 , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , -0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , -0.5 , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , -0.5 , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0.5 , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0.5 , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , -0.5 , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , -0.5 , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , -0.5 , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0.5 , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0.5 , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0.5 , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , -0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , -0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , -0.5 , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , -0.5 , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , -0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0.5 , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0.5 , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 1. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0. , 1. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0. , 1. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0. , 1. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , -0.5 , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , -0.5 , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0.5 , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0.5 , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0. , 1. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0. , 1. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , -0.5 , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , -0.5 , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , -0.5 , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , -0.5 , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. ], [ 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. ], [ 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. ], [ 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. ], [ 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , -0.5 , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , -0.5 , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0. , 0.5 , 0.5 , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0.5 , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0.5 , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0.5 , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. ], [ 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. ], [ 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. ], [ 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. ], [ 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 ], [ 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 ], [ 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0.5 , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 0.5 , 0.5 , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0. , 1. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , -0.5 , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , -0.5 , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , -0.5 , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , -0.5 , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. ], [ 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. ], [ 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 ], [ 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 ], [ 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 ], [ 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 ], [ 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. ], [ 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. ], [ 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , -0.5 , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , -0.5 , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , -0.5 , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , -0.5 , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , -0.5 , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , -0.5 , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , -0.5 , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , -0.5 , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. ], [ 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. ], [ 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 ], [ 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 ], [ 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 ], [ 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 ], [ 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. ], [ 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. ], [ 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , -0.5 , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , -0.5 , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , -0.5 , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , -0.5 , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. ], [ 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. ], [ 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. ], [ 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. ], [ 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 ], [ 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 ], [ 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 ], [ 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 ], [ 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 ], [ 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 ], [ 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 ], [ 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 ], [ 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. ], [ 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. ], [ 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. ], [ 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. ], [ 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 ], [ 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 ], [ 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0. , 0.5 , 0. , 0.5 , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0.5 , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0.5 , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0.5 , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. ], [ 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. ], [ 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 ], [ 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 ], [ 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 ], [ 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 ], [ 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. ], [ 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. ], [ 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0.5 , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0.5 , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0.5 , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0. , 0.5 , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0.5 , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0.5 , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0.5 , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0.5 , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. ], [ 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. ], [ 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 ], [ 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 ], [ 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 ], [ 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 ], [ 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. ], [ 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. ], [ 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0.5 , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0.5 , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0.5 , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 0.5 , 0.5 , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0. , 1. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, -0.25, 0.25, -0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25, 0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25, 0.25, 0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, 0.25, 0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, -0.25, -0.25, -0.25, -0.25], [ 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25], [ 0.5 , -0.5 , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , -0.5 , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , -0.5 , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , -0.5 , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , -0.5 , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 ], [ 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 ], [ 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. ], [ 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. ], [ 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. ], [ 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. ], [ 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 ], [ 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 ], [ 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , -0.5 , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , -0.5 , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , -0.5 , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , -0.5 , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , -0.5 , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , -0.5 , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , -0.5 , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. ], [ 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. ], [ 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 ], [ 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 ], [ 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 ], [ 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 ], [ 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. ], [ 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. ], [ 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , -0.5 , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , -0.5 , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , -0.5 , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , -0.5 , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , -0.5 , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , -0.5 , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 ], [ 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 ], [ 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 ], [ 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 ], [ 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , -0.5 , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , -0.5 , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , -0.5 , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. ], [ 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. ], [ 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 ], [ 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 ], [ 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 ], [ 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 ], [ 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. ], [ 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. ], [ 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 ], [ 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 ], [ 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 ], [ 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 ], [ 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , -0.5 ], [ 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0.5 ], [ 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 ], [ 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 ], [ 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 , 0. ], [ 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 , 0. ], [ 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , -0.5 ], [ 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. , 0.5 ], [ 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 ], [ 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 ], [ 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. ], [ 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. ], [ 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , -0.5 , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. , 0.5 , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0.5 , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0.5 , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0.5 , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 ], [ 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 ], [ 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 ], [ 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 ], [ 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. ], [ 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0.5 , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0.5 , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0.5 , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0.5 , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0.5 , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0.5 , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 , 0. ], [ 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 , 0. ], [ 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , -0.5 ], [ 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0.5 ], [ 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 ], [ 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 ], [ 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. ], [ 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. ], [ 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0.5 , 0. , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0.5 , 0. , 0.5 , 0. , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0. , 0.5 , 0. , 0.5 , 0. , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0.5 , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0.5 , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0.5 , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0.5 , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0.5 , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0.5 , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , -0.5 ], [ 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0. , 0. , 0.5 ], [ 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , -0.5 , 0. ], [ 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , -0.5 , 0.5 , 0. ], [ 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. ], [ 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. ], [ 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 ], [ 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 ], [ 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , -0.5 , 0. , 0. , 0. , 0. ], [ 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0.5 , 0. , 0. , 0.5 , 0. , 0. , 0. , 0. ], [ 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0.5 , 0. , 0. , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0.5 , 0. , 0. , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0.5 , 0.5 , -0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 0.5 , 0.5 , 0.5 , 0.5 , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ], [ 1. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ]]))

In [ ]:
#@title Verify AlphaEvolve's solution is a valid kissing configuration using exact integer arithmetic (takes 2 minutes).

def is_valid_kissing_configuration_exact_float(centers: np.ndarray) -> int:
  """Returns whether `centers` is a valid kissing configuration.

  First, converts all points to integers by multiplying by a large constant.
  Then, using exact integer arithmetic, verifies that all pairs x, y satisfy
  2 * <x,y> <= ||x|| ||y||, hence the angle between any two points is at least
  60 degrees, thus `centers` is a valid kissing configuration after normalizing.

  Args:
    centers: A set of points with floating-point coordinates. Shape (n, d).

  Returns:
    True if the angle between any two points in `centers` is at least 60 degrees.
  """
  (n, d) = centers.shape

  # Convert all points to fractions
  fraction_centers = np.empty(centers.shape, dtype=object)
  max_denominator = 1
  for i, j in itertools.product(range(n), range(d)):
    fraction_centers[i, j] = fractions.Fraction(centers[i, j])
    max_denominator = max((max_denominator), (fraction_centers[i, j].denominator))

  # The maximum denominator is already a power of 2 because floating-point
  # numbers are represented in base 2.
  scaling_factor = max_denominator

  # Multiply each fraction by `scaling_factor` and extract the numerators.
  # These will be our integer points for exact calculations.
  # We use dtype=object to support arbitrarily large Python integers.
  integer_centers = np.empty(centers.shape, dtype=object)
  norm_squared = np.zeros(n, dtype=object)
  for i, j in itertools.product(range(n), range(d)):
    # Multiplying a Fraction by an integer yields a Fraction. Since
    # scaling_factor is a multiple of the denominator, the new denominator
    # must be 1.
    assert (fraction_centers[i, j] * scaling_factor).denominator == 1
    integer_centers[i, j] = int((fraction_centers[i, j] * scaling_factor).numerator)
    norm_squared[i] += (integer_centers[i, j]) * (integer_centers[i, j])

  assert (norm_squared > 0).all(), 'One of the centers is zero ... this is not allowed!'

  # Perform exact integer calculations to check all pairs x, y satisfy 2 * <x,y> <= ||x|| ||y||.
  for i, j in itertools.combinations(range(n), 2):
    dot_product = sum(x_k * y_k for x_k, y_k in zip(integer_centers[i], integer_centers[j]))
    if (dot_product >= 0 and 4 * dot_product * dot_product > norm_squared[i] * norm_squared[j]):
      return False

  return True

if is_valid_kissing_configuration_exact_float(alphaevolve_solution):
  print(f'Verified (using exact computation with integers) that AlphaEvolve solution is a valid kissing configuration at dimension {alphaevolve_solution.shape[1]} with {alphaevolve_solution.shape[0]} spheres.')
else:
  print(f'Could not verify that the sphere packing is valid.')

In [ ]:
#@title Verify AlphaEvolve's point set has different signatures from all known point sets in https://dspace.mit.edu/handle/1721.1/153312 (takes 4 minutes)

def eigenvalues(point_set: np.ndarray) -> np.ndarray:
  distance_matrix = np.linalg.norm(point_set[:, None] - point_set[None, :], axis=2)
  return np.linalg.eigvalsh(distance_matrix)

alphaevolve_eigenvalues = eigenvalues(alphaevolve_solution)

print(f'There are {len(known_solutions)} solutions.', flush=True)

# If the eigenvalues are different, then the point clouds are not distance-isomorphic
# (one cannot be obtained from the other by a transformation that preserves distances).
for i, point_set in enumerate(known_solutions):
  distance = np.linalg.norm(eigenvalues(point_set) - alphaevolve_eigenvalues)
  print(f'Euclidean distance between AlphaEvolve solution signature and known solution #{i+1} signature: {distance}', flush=True)
  assert distance > 0.3, 'AlphaEvolve solution signature is similar to a known solution signature'

print('\nSince all distances > 0.3, the AlphaEvolve solution is distinct from known ones.')